# Category-Aware LLM Routing on TACO — Controlled Study
### Single self-contained pipeline: corpus → generation → grading → CV routing → policies → tables & figures

Companion to the HumanEval+ notebook, same protocol and same statistics, so the two runs are a
genuine replication pair. Run it top to bottom; every stage is checkpointed to disk and resumes
after a crash.

---

## What this fixes relative to the three original TACO notebooks

| Problem | Fix here |
|---|---|
| `category_route()` fit the route map on all problems, then scored it on the same problems | **10-fold cross-validated routing** and a CV best-single baseline |
| Graded on `inputs[:5]` only | **All provided cases** (cap recorded per problem), with a separate public/private split for cascade gating |
| "pass@1" was sampled at T=0.2 | Greedy for the headline pass@1; sampling only for pass@k |
| pass@5 = any-of-5 (biased down) | **Unbiased pass@k** (Chen et al., 2021) from `N_SAMPLES` samples |
| Cascades escalated on the same cases they were graded on | Gate on the first `PUBLIC_GATE_K` cases, grade on all cases — deploy-realistic and not circular |
| Token costs estimated as `chars/4` | Real tokenizer counts logged at generation time |
| No CIs, no multiplicity correction | Wilson + bootstrap CIs, exact McNemar, BH + Holm |
| Many-to-one tag→category mapping assumed correct | Multi-tag rate reported + **priority-permutation sensitivity** (Table 8) |
| StarCoder2 (base) mixed in with instruct models | Completion-style prompting for base models, excluded from the routing pool |
| Phase-2 reflection agent evaluated against nothing | Equal-budget **repair-vs-resample** arm — TACO's failing input/expected/actual is richer feedback than HumanEval's |
| Hardcoded HF token | Environment variable / Kaggle secret only |
| ~40–80 GPU-hours with `model.generate` per problem | **vLLM backend**, batched; HF retained as a fallback |

> **Rotate your HuggingFace token.** The original `LLM_Routing_Pipeline.ipynb` and
> `Phase2_Budget_Coder.ipynb` contain a live token in the config cell. Revoke it at
> huggingface.co/settings/tokens and use `HF_TOKEN` in the environment instead.

## Why TACO is the harder half of the story

Your best model sat near 9.6% here. That is *good* for a routing study — an unsaturated
benchmark preserves per-category variance — but it puts the burden on you to show the low
numbers are the benchmark rather than your harness. Two tables do that work: **Table 2b**
(difficulty gradient — easy items must score visibly higher) and **Table 2c** (failure
composition — failures should be wrong answers, not syntax errors or missing functions). If
easy problems also sit near zero, fix prompting before writing a single sentence of the paper.

## Stage checkpoints (all under `RESULTS_DIR`)
```
corpus.json          problems, categories, difficulty, test cases
gen/<model>.jsonl    raw generations, appended per problem
grade/<model>.jsonl  execution verdicts, appended per problem
repair/<model>.jsonl repair-arm attempts
tables/  figures/    paper artifacts
```

## How to run

**Step 1 — smoke test (~10 min, no GPU).** `DRY_RUN = True`, run everything. This exercises the
corpus builder, both executor modes, all statistics and all figures on mock generations.

**Step 2 — the sweep.** `DRY_RUN = False`, run §0 → §4. GPU only, fully resumable: interrupt it,
restart the kernel, re-run, it continues from the last completed problem.

**Step 3 — analysis.** §5 → §9 any time. CPU-only, about a minute, re-runnable as often as you
like since it reads only the on-disk checkpoints.

Nothing to attach: TACO is pulled from the Hub and the taxonomy is built from its tags.

In [ ]:
# ── 0.1 · Install ─────────────────────────────────────────────────────────────
import subprocess, sys, importlib

CORE = ["transformers>=4.45.0", "accelerate>=0.34.0", "torch", "datasets",
        "huggingface_hub", "tqdm", "pandas", "numpy", "scipy", "statsmodels",
        "matplotlib", "sentencepiece", "protobuf"]
TRY_INSTALL_VLLM = True          # 10-20x faster; set False if it fights your CUDA build

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet"] + CORE, check=False)
if TRY_INSTALL_VLLM and importlib.util.find_spec("vllm") is None:
    print("installing vllm (a few minutes) ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "vllm"], check=False)

HAS_VLLM = importlib.util.find_spec("vllm") is not None
print(f"vllm available: {HAS_VLLM}")
if not HAS_VLLM:
    print("NOTE: falling back to transformers generate() -- much slower. See BACKEND in the config.")

In [ ]:
# ── 0.2 · CONFIG — the only cell you normally edit ────────────────────────────
import os, math, json, re
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
# Must be set BEFORE torch initialises CUDA. Expandable segments greatly reduce the
# fragmentation that makes long generation sweeps OOM after running fine for an hour.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
try:
    import torch; HAS_TORCH = True
except Exception:
    torch = None; HAS_TORCH = False

RESULTS_DIR = "taco_run/"        # NEW directory: old benchmark_results/ used 5-case grading and
                                 # sampled pass@1. Those numbers are NOT comparable -- do not mix.
for sub in ("", "gen", "grade", "repair", "tables", "figures", "runners"):
    os.makedirs(os.path.join(RESULTS_DIR, sub), exist_ok=True)

# --- model cache location -----------------------------------------------------
# 13 models is ~250 GB. The default cache is on C:, which fills up and leaves TRUNCATED
# downloads that then fail with a confusing "does not appear to have files named ..." error.
# Point this at a drive with room BEFORE the first download.
HF_CACHE_DIR = os.environ.get("HF_HOME", r"G:\hf_cache")   # set to "" for the default
                                                          # ~/.cache/huggingface location
if HF_CACHE_DIR:
    os.environ["HF_HOME"] = HF_CACHE_DIR
    os.environ["HF_HUB_DISABLE_SYMLINKS"] = "1"   # needed on Windows / mounted drives

# --- credentials --------------------------------------------------------------
# Never hard-code a token. Kaggle: Add-ons -> Secrets. Colab: userdata. Shell: export HF_TOKEN=...
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
if not HF_TOKEN:
    print("WARNING: no HF_TOKEN found. Gated repos (Llama-3.1, CodeGemma) will be skipped.")

# --- run modes ----------------------------------------------------------------
DRY_RUN     = False    # True = mock generations, no GPU (full pipeline smoke test)
FRESH_START = False    # True = rebuild corpus.json
QUICK_TEST  = False; QUICK_CAP  = 5
TRIAL_MODE  = False; TRIAL_TOTAL = 24

# --- backend ------------------------------------------------------------------
BACKEND        = "auto"           # "auto" | "vllm" | "hf"
PRECISION_MODE = "uniform_bf16"   # "uniform_bf16" (recommended) | "as_listed"
GPU_MEM_UTIL   = 0.90
MAX_MODEL_LEN  = 4096

# --- HF backend performance (ignored when the vLLM backend is active) ---------
# vLLM has no Windows build, so on Windows the HF path IS the pipeline.
ATTN_IMPL     = "sdpa"   # "sdpa" = PyTorch native, fast, Windows-friendly. "eager" is the slow
                         # fallback and warns on sliding-window models. "flash_attention_2" is
                         # Linux-only; the engine falls back automatically if unavailable.
HF_MAX_SEQS   = 8        # CEILING on concurrent sequences. The engine measures free VRAM
                         # after loading each model and auto-sizes below this, so you normally
                         # do not touch it. Raise it only if the printed plan shows spare room.
GPU_MEM_FRACTION = 0.92  # hard cap on the fraction of the card this process may allocate.
                         # Critical on Windows: without it the driver silently spills to shared
                         # system RAM (Task Manager -> "Shared GPU memory") instead of raising
                         # OOM, which is ~70x slower and looks like a hang, not an error.
PROMPT_TOKEN_BUDGET = 1024   # fallback only. v4 measures the ACTUAL longest prompt in each
                             # slice and sizes the batch from that. The old fixed 1024 estimate
                             # under-counted KV by up to 2.4x on long TACO statements (prompts
                             # truncate at MAX_MODEL_LEN - MAX_NEW_TOKENS = 3328 tokens), which
                             # is what drove the 13B OOMs late in the length-sorted sweep.
KV_SAFETY_FRAC = 0.55        # share of free VRAM the KV cache may claim; the rest absorbs
                             # activations, the logits tensor and allocator fragmentation
HF_BATCH_SIZE = HF_MAX_SEQS   # backwards-compatible alias

# --- corpus / taxonomy --------------------------------------------------------
MIN_PROBLEMS_PER_CATEGORY = 10
MAX_PROBLEMS_PER_CATEGORY = 80    # keeps categories comparable and the sweep affordable
MAX_TEST_CASES  = 10              # grade on up to this many cases (pre-cap count is recorded)
MAX_INPUT_CHARS = 20_000          # skip pathological mega-inputs
MAX_QUESTION_CHARS = 6_000        # truncate very long statements in the prompt
PUBLIC_GATE_K   = 2               # cases a cascade may look at before escalating.
                                  # Grading always uses ALL cases. Gating on the grading set
                                  # is the single most common way to fake a good cascade.

# --- generation protocol ------------------------------------------------------
GLOBAL_SEED    = 0
GREEDY_TEMP    = 0.0
SAMPLE_TEMP    = 0.8
TOP_P          = 0.95
N_SAMPLES      = 10               # set 6 to cut ~40% of GPU time
PASS_KS        = [1, 3, 5, 10]     # ks below are clamped to N_SAMPLES automatically
MAX_NEW_TOKENS = 768              # competitive-programming solutions run long

# --- grading ------------------------------------------------------------------
PER_CASE_TIMEOUT = 6              # seconds per test case (inside the child process)
EXEC_TIMEOUT     = 45             # hard wall-clock ceiling per candidate, all cases
N_EXEC_WORKERS   = min(32, max(2, (os.cpu_count() or 4) - 1))   # capped: beyond ~32
                                       # concurrent subprocesses this stops helping

# --- statistics ---------------------------------------------------------------
CV_FOLDS = 10
N_BOOT   = 5000
ALPHA    = 0.05
MIN_CAT_N_FOR_TESTS = 10
ROUTE_MIN_MARGIN    = 0.0
TAXONOMY_PERMS      = 20          # priority-order permutations for Table 8

# --- optional arms ------------------------------------------------------------
RUN_REPAIR_ARM  = True
REPAIR_ATTEMPTS = 3               # total generations incl. the reused greedy one

# --- generation integrity -----------------------------------------------------
# v4: never checkpoint a record whose generations came back empty. In v3 an OOM wrote
# {"greedy_code": "", "sample_codes": ["" x 10]} to the JSONL, which marked the problem
# "done" forever -- so the resume skipped it and the model silently scored 0 on it.
SKIP_EMPTY_RECORDS = True

# --- models -------------------------------------------------------------------
MODELS = [
 {"name":"Qwen2.5-Coder-7B",   "repo":"Qwen/Qwen2.5-Coder-7B-Instruct",       "params":"7.6B", "in4bit":False,"base":False,"price":1.0,"enabled":True},
 {"name":"DeepSeek-Coder-6.7B","repo":"deepseek-ai/deepseek-coder-6.7b-instruct","params":"6.7B","in4bit":False,"base":False,"price":0.9,"enabled":True},
 {"name":"Mistral-7B",         "repo":"mistralai/Mistral-7B-Instruct-v0.3",   "params":"7.2B", "in4bit":False,"base":False,"price":1.0,"enabled":True},
 {"name":"LLaMA-3.1-8B",       "repo":"meta-llama/Llama-3.1-8B-Instruct",     "params":"8.0B", "in4bit":False,"base":False,"price":1.1,"enabled":True},
 {"name":"Phi-3.5-Mini",       "repo":"microsoft/Phi-3.5-mini-instruct",      "params":"3.8B", "in4bit":False,"base":False,"price":0.5,"enabled":True},
 {"name":"StarCoder2-7B",      "repo":"bigcode/starcoder2-7b",                "params":"7.2B", "in4bit":False,"base":True ,"price":1.0,"enabled":True},
 {"name":"CodeLlama-7B",       "repo":"codellama/CodeLlama-7b-Instruct-hf",   "params":"7.0B", "in4bit":False,"base":False,"price":1.0,"enabled":True},
 {"name":"CodeGemma-7B",       "repo":"google/codegemma-7b-it",               "params":"7.0B", "in4bit":True ,"base":False,"price":1.0,"enabled":True},
 {"name":"Granite-4.1-8B",     "repo":"ibm-granite/granite-4.1-8b",           "params":"8.1B", "in4bit":True ,"base":False,"price":1.1,"enabled":True},
 {"name":"Qwen2.5-Coder-3B",   "repo":"Qwen/Qwen2.5-Coder-3B-Instruct",       "params":"3.1B", "in4bit":False,"base":False,"price":0.4,"enabled":True},
 {"name":"Qwen2.5-Coder-14B",  "repo":"Qwen/Qwen2.5-Coder-14B-Instruct",      "params":"14.7B","in4bit":True ,"base":False,"price":2.0,"enabled":True},
 {"name":"StarCoder2-15B",     "repo":"bigcode/starcoder2-15b",               "params":"15B",  "in4bit":True ,"base":True ,"price":2.0,"enabled":True},
 {"name":"CodeLlama-13B",      "repo":"codellama/CodeLlama-13b-Instruct-hf",  "params":"13B",  "in4bit":True ,"base":False,"price":1.8,"enabled":True},
]
# base=True  -> completion-style prompt, and excluded from the routing pool. A base model under
#               a chat template produces a fake "bad at category X" result.
# PRECISION_MODE defaults to uniform bf16 so per-category winners are a statement about models,
# not about quantization. The vLLM backend is bf16-only regardless.
# The preflight cell below verifies every repo and disables missing/gated ones.

# budgets used by the policy comparison, clamped so that lowering N_SAMPLES degrades the
# analysis gracefully instead of filling the resample rows with NaN
RESAMPLE_K  = min(5, N_SAMPLES)                  # width arm budget
REPAIR_K    = min(REPAIR_ATTEMPTS, N_SAMPLES)    # blind-resample baseline for the repair arm
PASS_KS     = sorted(set([k for k in PASS_KS if k <= N_SAMPLES] + [1, RESAMPLE_K, REPAIR_K]))

PRICE   = {m["name"]: m["price"] for m in MODELS}
IS_BASE = {m["name"]: m["base"] for m in MODELS}
if HAS_TORCH and torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print(f"mode      : {'DRY RUN' if DRY_RUN else 'LIVE'}"
      f"{' | TRIAL' if TRIAL_MODE else ''}{' | QUICK' if QUICK_TEST else ''}")
print(f"protocol  : greedy@{GREEDY_TEMP} + {N_SAMPLES}@T={SAMPLE_TEMP}/top-p{TOP_P}, unbiased pass@{PASS_KS}")
print(f"budgets   : width arm = resample{RESAMPLE_K}, repair baseline = blind resample@{REPAIR_K}")
print(f"grading   : all cases up to {MAX_TEST_CASES} | cascade gate = first {PUBLIC_GATE_K} cases")
print(f"models    : {sum(m['enabled'] for m in MODELS)} enabled "
      f"({sum(m['enabled'] and m['base'] for m in MODELS)} base, excluded from routing)")

In [ ]:
# ── 0.3 · Preflight: GPU, disk, repo availability ─────────────────────────────
import shutil
if HAS_TORCH and torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name} | {p.total_memory/1024**3:.1f} GB | torch {torch.__version__} | cuda {torch.version.cuda}")
else:
    print("No GPU visible -- generation only works with DRY_RUN=True.")
# Other notebooks/kernels holding GPU memory are the single most common cause of
# "it worked last week": your weights fit, but only into what is actually free.
try:
    _q = subprocess.run(["nvidia-smi", "--query-compute-apps=pid,used_memory",
                         "--format=csv,noheader"], capture_output=True, text=True, timeout=20)
    _apps = [l for l in _q.stdout.strip().split("\n") if l.strip()]
    if _apps:
        print("\nProcesses already using the GPU:")
        for l in _apps: print("   ", l)
        _mine = os.getpid()
        if any(str(_mine) != l.split(",")[0].strip() for l in _apps):
            print("  ^^ CLOSE any other notebook kernels before starting the sweep. Every one of")
            print("     these is holding VRAM that this run needs (Kernel -> Shut Down Kernel,")
            print("     or Jupyter's 'Running' tab -> shut down all).")
except Exception:
    pass
if HAS_TORCH and torch.cuda.is_available():
    _free, _total = torch.cuda.mem_get_info()
    print(f"\nVRAM free right now: {_free/1024**3:.1f} / {_total/1024**3:.1f} GB")
    if _free < 0.8 * _total:
        print("  WARNING: a large part of the card is already occupied. Free it before running "
              "section 4, or the sweep will crawl or die.")
_cache = os.environ.get("HF_HOME") or os.path.expanduser("~/.cache/huggingface")
_hub = os.path.join(_cache, "hub")
if os.path.isdir(_hub):
    _cached = sorted(d.replace("models--", "").replace("--", "/")
                     for d in os.listdir(_hub) if d.startswith("models--"))
    print(f"\nModels already cached ({len(_cached)}) at {_hub}:")
    for _m in _cached: print(f"    {_m}")
elif HF_CACHE_DIR:
    print(f"\nNOTE: {_hub} does not exist yet -- every model will be downloaded there.")
try:
    _cfree = shutil.disk_usage(os.path.dirname(_cache) if os.path.exists(os.path.dirname(_cache))
                               else ".").free / 1024**3
    _need = 25 * sum(1 for m in MODELS if m["enabled"])
    print(f"\nModel cache: {_cache}")
    print(f"  free there: {_cfree:.0f} GB | ~{_need} GB needed for {sum(1 for m in MODELS if m['enabled'])} models")
    if _cfree < _need:
        print("  WARNING: not enough room. Downloads will truncate and the model will then fail")
        print("  with 'does not appear to have files named ...'. Set HF_CACHE_DIR in the config")
        print("  cell to a bigger drive and restart the kernel, or clear space first.")
except Exception:
    pass
print(f"Disk free: {shutil.disk_usage('.').free/1024**3:.0f} GB "
      f"(~25 GB of HF cache per 13B model; point HF_HOME at a large volume)")

if not DRY_RUN:
    # v4: model_info() only reads repo METADATA, which HF serves for gated repos you are not
    # authorised to download from. CodeGemma-7B therefore passed this check and then died an
    # hour into the sweep with a 403 on config.json. Fetch the real file instead.
    from huggingface_hub import hf_hub_download
    from huggingface_hub.utils import GatedRepoError, RepositoryNotFoundError
    for m in MODELS:
        if not m["enabled"]:
            continue
        try:
            hf_hub_download(m["repo"], "config.json", token=HF_TOKEN or None)
            print(f"  ok       {m['name']:<22} {m['repo']}")
        except GatedRepoError:
            m["enabled"] = False
            print(f"  DISABLED {m['name']:<22} GATED -- accept the licence at")
            print(f"           https://huggingface.co/{m['repo']}  (sign in with the SAME")
            print(f"           account your HF_TOKEN belongs to), then re-run this cell.")
        except RepositoryNotFoundError:
            m["enabled"] = False
            print(f"  DISABLED {m['name']:<22} {m['repo']}  (repo not found)")
        except Exception as e:
            s = str(e)
            m["enabled"] = False
            if "403" in s or "401" in s:
                print(f"  DISABLED {m['name']:<22} GATED/unauthorised --")
                print(f"           https://huggingface.co/{m['repo']}")
            else:
                print(f"  DISABLED {m['name']:<22} {m['repo']}  ({type(e).__name__}: {s[:70]})")
ENABLED     = [m["name"] for m in MODELS if m["enabled"]]
MAIN_MODELS = [m["name"] for m in MODELS if m["enabled"] and not m["base"]]
print(f"\nwill benchmark ({len(ENABLED)}): {ENABLED}")
print(f"routing pool  ({len(MAIN_MODELS)}): {MAIN_MODELS}")

## 1 · Corpus and taxonomy

TACO problems come in two shapes and the harness must respect both: **stdin** problems (read
with `input()`, print to stdout) and **call-based** problems (`fn_name` in `input_output`, args
passed positionally). Mixing them up silently scores every call-based problem as a failure.

The tag→category mapping is many-to-one and resolved by a fixed priority order. That choice is
a modelling decision, not a fact, so §1.2 reports the multi-tag rate and §7.6 re-runs the whole
routing analysis under random permutations of the priority order.

In [ ]:
# ── 1.1 · Taxonomy definition ─────────────────────────────────────────────────
CATEGORY_PRIORITY = [
    "Dynamic Programming", "Graph Algorithms", "String Manipulation", "Mathematical",
    "Greedy", "Sorting & Searching", "Data Structures", "Recursion & Backtracking",
    "Complete Search",
]
CATEGORY_MAP = {
    "Dynamic Programming":      ["dynamic programming", "dp", "memoization", "knapsack", "bitmask dp",
                                 "digit dp", "longest common subsequence", "bitmasks", "meet in the middle"],
    "Graph Algorithms":         ["graphs", "graph", "bfs", "dfs", "dfs and similar", "shortest path",
                                 "shortest paths", "minimum spanning tree", "topological sort",
                                 "strongly connected components", "dijkstra", "floyd-warshall",
                                 "flows", "trees", "game theory", "2-sat"],
    "String Manipulation":      ["strings", "string", "pattern matching", "suffix array",
                                 "string suffix structures", "kmp", "z-algorithm", "trie", "palindrome",
                                 "anagram", "hashing", "suffix automaton", "aho-corasick",
                                 "expression parsing"],
    "Mathematical":             ["math", "mathematics", "number theory", "combinatorics",
                                 "modular arithmetic", "prime", "primes", "gcd", "lcm", "geometry",
                                 "probability", "matrices", "matrix exponentiation", "fft",
                                 "chinese remainder theorem"],
    "Greedy":                   ["greedy", "greedy algorithm", "constructive algorithms",
                                 "interval scheduling", "activity selection"],
    "Sorting & Searching":      ["sorting", "sort", "binary search", "search", "two pointers",
                                 "two-pointers", "sliding window", "divide and conquer"],
    "Data Structures":          ["data structures", "stack", "queue", "heap", "priority queue",
                                 "segment tree", "fenwick tree", "binary indexed tree", "disjoint set",
                                 "union find", "dsu", "deque", "sparse table", "monotonic stack",
                                 "monotonic queue"],
    "Recursion & Backtracking": ["recursion", "backtracking", "permutations", "combinations", "n-queens"],
    "Complete Search":          ["complete search", "brute force", "exhaustive search", "simulation",
                                 "implementation", "ad hoc"],
}

def normalise_tag(t): return re.sub(r"\s+", " ", str(t).strip().lower())

def parse_tags(raw):
    if isinstance(raw, str):
        try:
            raw = json.loads(raw.replace("'", '"'))
        except Exception:
            raw = [raw]
    return [normalise_tag(t) for t in (raw or []) if t]

def matched_categories(tags):
    """Every taxonomy category this problem's tags hit (used for the multi-tag audit)."""
    return [c for c in CATEGORY_PRIORITY
            if any(kw in t for t in tags for kw in CATEGORY_MAP[c])]

def assign_category(hits, priority=None):
    for c in (priority or CATEGORY_PRIORITY):
        if c in hits:
            return c
    return None

SHORT = {"Dynamic Programming": "DP", "Graph Algorithms": "Graph", "String Manipulation": "String",
         "Mathematical": "Math", "Greedy": "Greedy", "Sorting & Searching": "Sort/Search",
         "Data Structures": "DS", "Recursion & Backtracking": "Recursion",
         "Complete Search": "C.Search"}
sh = lambda c: SHORT.get(c, c[:10])
print(f"taxonomy: {len(CATEGORY_PRIORITY)} categories, "
      f"{sum(len(v) for v in CATEGORY_MAP.values())} keywords")

In [ ]:
# ── 1.2 · Build the corpus from the TACO test split ───────────────────────────
import random
import numpy as np, pandas as pd
from collections import defaultdict, Counter

CORPUS_PATH = os.path.join(RESULTS_DIR, "corpus.json")

def _tqdm(it, desc):
    try:
        from tqdm.auto import tqdm; return tqdm(it, desc=desc)
    except Exception:
        return it

def _mock_corpus():
    """DRY_RUN corpus with the same schema: alternating stdin / call-based problems."""
    out = {}
    for ci, cat in enumerate(CATEGORY_PRIORITY[:6]):
        probs = []
        for i in range(12):
            fn = (i % 2 == 1)
            probs.append({
                "question": f"[mock {cat} #{i}] Given two integers, output their sum.",
                "starter": "def solve(a, b):" if fn else "",
                "inputs": [[1, 2], [10, 20], [0, 5]] if fn else ["1 2", "10 20", "0 5"],
                "outputs": [3, 30, 5] if fn else ["3", "30", "5"],
                "n_cases_full": 3, "fn_name": "solve" if fn else None,
                "difficulty": ["easy", "medium", "hard"][i % 3],
                "category": cat, "tags": [cat.lower()], "n_tag_hits": 1 + (i % 2)})
        out[cat] = probs
    return out

def _taco_rows():
    """BAAI/TACO ships a loading script (TACO.py). datasets>=3.0 refuses to run scripts, so
    fall back to the Hub's auto-converted parquet export, which needs no script."""
    try:
        from datasets import load_dataset
        kw = {"split": "test"}
        if HF_TOKEN: kw["token"] = HF_TOKEN
        ds = load_dataset("BAAI/TACO", **kw)
        print(f"    loaded via datasets ({len(ds)} rows)")
        return ds
    except Exception as e:
        print(f"    load_dataset unavailable: {type(e).__name__}: {str(e)[:120]}")
        print("    falling back to the parquet export ...")
    from huggingface_hub import HfApi, hf_hub_download
    api = HfApi(token=HF_TOKEN or None)
    for rev in ("refs/convert/parquet", "main"):
        try:
            files = api.list_repo_files("BAAI/TACO", repo_type="dataset", revision=rev)
        except Exception:
            continue
        pq = [f for f in files if f.endswith(".parquet") and "/test/" in f]
        if not pq:
            pq = [f for f in files if f.endswith(".parquet")
                  and "test" in os.path.basename(f).lower()]
        if not pq:
            continue
        # the export has one directory per config (ALL, EASY, MEDIUM, ...). Take ALL, or a
        # single config, so the same problem is not counted once per difficulty config.
        cfgs = sorted({f.split("/")[0] for f in pq})
        pick = "ALL" if "ALL" in cfgs else cfgs[0]
        pq = [f for f in pq if f.startswith(pick + "/")] or pq
        print(f"    revision {rev}, config '{pick}', {len(pq)} parquet file(s)")
        frames = []
        for f in _tqdm(pq, "downloading parquet"):
            p = hf_hub_download("BAAI/TACO", f, repo_type="dataset", revision=rev,
                                token=HF_TOKEN or None)
            frames.append(pd.read_parquet(p))
        df = pd.concat(frames, ignore_index=True)
        print(f"    loaded via parquet ({len(df)} rows, config '{pick}')")
        return df.to_dict("records")
    raise RuntimeError(
        "Could not load BAAI/TACO. Either pin datasets:  pip install 'datasets<3.0.0'  "
        "or fetch the parquet files manually from "
        "https://huggingface.co/datasets/BAAI/TACO/tree/refs%2Fconvert%2Fparquet")

def load_taco():
    print("loading BAAI/TACO test split ...")
    ds = _taco_rows()
    print(f"raw problems: {len(ds)}")
    by_cat, stats = defaultdict(list), Counter()
    for row in _tqdm(ds, "parsing"):
        tags = parse_tags(row.get("tags", "[]"))
        hits = matched_categories(tags)
        cat = assign_category(hits)
        if cat is None:
            stats["no_category"] += 1; continue
        io = row.get("input_output", "{}")
        if isinstance(io, str):
            try: io = json.loads(io)
            except Exception: stats["bad_io"] += 1; continue
        inputs, outputs, fn = io.get("inputs", []), io.get("outputs", []), io.get("fn_name")
        if not inputs or len(inputs) != len(outputs):
            stats["bad_io"] += 1; continue
        inputs, outputs = inputs[:MAX_TEST_CASES], outputs[:MAX_TEST_CASES]
        if any(len(str(i)) > MAX_INPUT_CHARS for i in inputs):
            stats["input_too_large"] += 1; continue
        try:                                    # the whole record must survive a JSON round-trip
            json.dumps([inputs, outputs])
        except Exception:
            stats["unserialisable"] += 1; continue
        stats["fn" if fn else "stdin"] += 1
        diff = str(row.get("difficulty", "unknown")).strip().lower()
        by_cat[cat].append({
            "question": row["question"].strip()[:MAX_QUESTION_CHARS],
            "starter": (row.get("starter_code") or "").strip(),
            "inputs": inputs, "outputs": outputs, "n_cases_full": len(io.get("inputs", [])),
            "fn_name": fn,
            # TACO ships difficulty in several spellings/cases across revisions
            "difficulty": diff if diff in ("easy", "medium", "medium_hard", "hard",
                                           "very_hard") else "unknown",
            "category": cat, "tags": tags, "n_tag_hits": len(hits)})
    rnd = random.Random(GLOBAL_SEED)
    corpus = {}
    for cat, probs in sorted(by_cat.items()):
        rnd.shuffle(probs)
        keep = len(probs) >= MIN_PROBLEMS_PER_CATEGORY
        print(f"  {cat:<28}{len(probs):>5}  {'keep' if keep else 'SKIP (below minimum)'}")
        if keep:
            corpus[cat] = probs[:MAX_PROBLEMS_PER_CATEGORY]
    json.dump(dict(stats), open(os.path.join(RESULTS_DIR, "corpus_ingest_stats.json"), "w"), indent=2)
    print(f"ingest stats: {dict(stats)}")
    return corpus

if os.path.exists(CORPUS_PATH) and not FRESH_START:
    CORPUS = json.load(open(CORPUS_PATH))
    print(f"loaded corpus.json ({sum(len(v) for v in CORPUS.values())} problems)")
else:
    CORPUS = _mock_corpus() if DRY_RUN else load_taco()
    json.dump(CORPUS, open(CORPUS_PATH, "w"))
    print(f"built corpus.json ({sum(len(v) for v in CORPUS.values())} problems)")

if QUICK_TEST:
    CORPUS = {c: p[:QUICK_CAP] for c, p in CORPUS.items()}
if TRIAL_MODE:
    picked, ptr = [], 0
    while len(picked) < TRIAL_TOTAL and ptr < max(len(v) for v in CORPUS.values()):
        for c in CORPUS:
            if ptr < len(CORPUS[c]) and len(picked) < TRIAL_TOTAL:
                picked.append((c, ptr))
        ptr += 1
    keep = defaultdict(list)
    for c, i in picked: keep[c].append(CORPUS[c][i])
    CORPUS = dict(keep)

PROBS = {f"{c}::{i:03d}": p for c in CORPUS for i, p in enumerate(CORPUS[c])}
KEYS_ALL = sorted(PROBS)
print(f"\ncorpus in use: {len(PROBS)} problems / {len(CORPUS)} categories")
for c in sorted(CORPUS, key=lambda c: -len(CORPUS[c])):
    fn = sum(1 for p in CORPUS[c] if p["fn_name"])
    print(f"  {c:<28}{len(CORPUS[c]):>4}   stdin:{len(CORPUS[c])-fn:<4} call-based:{fn}")

In [ ]:
# ── 1.3 · Taxonomy and corpus audit (report these in the paper) ───────────────
multi   = [p for p in PROBS.values() if p["n_tag_hits"] > 1]
cases   = [len(p["inputs"]) for p in PROBS.values()]
capped  = sum(1 for p in PROBS.values() if p["n_cases_full"] > len(p["inputs"]))
difficulty_mix = Counter(p["difficulty"] for p in PROBS.values())
type_mix = Counter("call-based" if p["fn_name"] else "stdin" for p in PROBS.values())

print(f"multi-tag problems (matched >1 category, resolved by priority): "
      f"{len(multi)}/{len(PROBS)} ({len(multi)/max(len(PROBS),1):.1%})")
print(f"test cases per problem: median {int(np.median(cases))}, "
      f"p10 {int(np.percentile(cases,10))}, p90 {int(np.percentile(cases,90))}")
print(f"problems whose case list was capped at {MAX_TEST_CASES}: {capped}")
print(f"difficulty mix : {dict(difficulty_mix)}")
print(f"problem types  : {dict(type_mix)}")

json.dump({"multi_tag_rate": len(multi) / max(len(PROBS), 1),
           "median_cases": float(np.median(cases)), "capped_problems": int(capped),
           "difficulty_mix": dict(difficulty_mix), "type_mix": dict(type_mix),
           "priority": CATEGORY_PRIORITY},
          open(os.path.join(RESULTS_DIR, "taxonomy_audit.json"), "w"), indent=2)
print("\nREAD THIS: a high multi-tag rate means the category label is partly arbitrary. "
      "Report it alongside Table 8 (priority-permutation sensitivity) rather than burying it.")

## 2 · Sandbox

One subprocess per candidate, grading all cases in one go and returning a **per-case** pass
vector. That vector is what makes an honest cascade possible: the gate looks only at the first
`PUBLIC_GATE_K` entries, while success is the conjunction over all of them — no second execution
pass and no leakage from the grading set into the escalation decision.

Per-case timeouts are enforced inside the child with `SIGALRM`, so one infinite loop costs you
one case rather than the whole candidate. The subprocess wall-clock timeout is the backstop.

In [ ]:
# ── 2.1 · Runner script on disk (no source-formatting or escaping games) ──────
RUNNER_DIR = os.path.join(RESULTS_DIR, "runners")

_TACO_RUNNER = r"""
import json, sys, os, io, copy, contextlib, threading, _thread
try:
    import resource                      # POSIX only
except ImportError:
    resource = None                      # Windows

def _limits():
    if resource is not None:
        try: resource.setrlimit(resource.RLIMIT_AS, (6 * 1024**3, 6 * 1024**3))
        except Exception: pass
    sys.setrecursionlimit(20000)

class _TLE(Exception): pass

class _Deadline:
    # Cross-platform per-case timeout. signal.SIGALRM does not exist on Windows, so a watchdog
    # thread raises KeyboardInterrupt in the main thread instead. This interrupts any pure-Python
    # loop on both platforms; a call blocked in C is caught by the outer subprocess timeout.
    def __init__(self):
        self.timer = None
        self.fired = False
    def start(self, secs):
        self.fired = False
        def _fire():
            self.fired = True
            _thread.interrupt_main()
        self.timer = threading.Timer(secs, _fire)
        self.timer.daemon = True
        self.timer.start()
    def stop(self):
        if self.timer is not None:
            self.timer.cancel()
            self.timer = None

_DL = _Deadline()

def _flat(v):
    # normalise an expected/actual value to a list of whitespace-stripped tokens
    if isinstance(v, (list, tuple)):
        parts = []
        for x in v: parts.extend(_flat(x))
        return parts
    s = "" if v is None else str(v)
    return [tok for tok in s.replace("\r", "\n").split() if tok != ""]

def _tok_eq(a, b):
    if a == b: return True
    try:
        fa, fb = float(a), float(b)
        return abs(fa - fb) <= 1e-6 * max(1.0, abs(fa), abs(fb))
    except Exception:
        pass
    return a.strip().lower() == b.strip().lower()

def _out_eq(got, exp):
    ga, ea = _flat(got), _flat(exp)
    if len(ga) != len(ea): return False
    return all(_tok_eq(x, y) for x, y in zip(ga, ea))

def _val_eq(a, b):
    if isinstance(a, bool) or isinstance(b, bool): return bool(a) is bool(b)
    if isinstance(a, (int, float)) and isinstance(b, (int, float)):
        try: return abs(a - b) <= 1e-6 * max(1.0, abs(float(a)), abs(float(b)))
        except Exception: return a == b
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_val_eq(x, y) for x, y in zip(a, b))
    if isinstance(a, dict) and isinstance(b, dict):
        return set(a) == set(b) and all(_val_eq(a[k], b[k]) for k in a)
    return a == b

def _expected_variants(exp):
    # TACO wraps call-based expectations inconsistently: [v] and v both occur
    out = [exp]
    if isinstance(exp, list) and len(exp) == 1: out.append(exp[0])
    return out

def main():
    _limits()
    pl = json.load(open(sys.argv[1]))
    code, mode, fn_name = pl["code"], pl["mode"], pl.get("fn_name")
    tests, per_case = pl["tests"], pl["per_case_timeout"]
    buf = io.StringIO()
    case_pass, first, stage = [], None, ""

    if mode == "fn":
        ns = {"__name__": "__solution__"}
        try:
            _DL.start(per_case)
            with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
                exec(compile(code, "<sol>", "exec"), ns)
            _DL.stop()
        except BaseException as e:
            _DL.stop()
            sys.stdout.write(json.dumps({"case_pass": [False] * len(tests), "stage": "solution_exec",
                "first": {"type": "runtime_error", "err": (type(e).__name__ + ": " + str(e))[:200],
                          "args": "", "expected": "", "got": ""}}))
            sys.stdout.flush(); os._exit(0)
        fn = ns.get(fn_name)
        if not callable(fn):
            sys.stdout.write(json.dumps({"case_pass": [False] * len(tests), "stage": "missing_fn",
                "first": {"type": "missing_fn", "err": "function " + str(fn_name) + " not defined",
                          "args": "", "expected": "", "got": ""}}))
            sys.stdout.flush(); os._exit(0)
        for args, exp in tests:
            call_args = args if isinstance(args, list) else [args]
            try:
                _DL.start(per_case)
                with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
                    got = fn(*copy.deepcopy(call_args))
                _DL.stop()
            except (KeyboardInterrupt, _TLE):
                _DL.stop(); case_pass.append(False)
                if first is None:
                    first = {"type": "tle", "err": "case timed out", "args": repr(args)[:200],
                             "expected": repr(exp)[:200], "got": ""}
                continue
            except BaseException as e:
                _DL.stop(); case_pass.append(False)
                if first is None:
                    first = {"type": "runtime_error", "err": (type(e).__name__ + ": " + str(e))[:200],
                             "args": repr(args)[:200], "expected": repr(exp)[:200], "got": ""}
                continue
            ok = any(_val_eq(got, cand) or _out_eq(got, cand) for cand in _expected_variants(exp))
            case_pass.append(ok)
            if not ok and first is None:
                first = {"type": "wrong_answer", "err": "", "args": repr(args)[:200],
                         "expected": repr(exp)[:200], "got": repr(got)[:200]}
    else:
        for inp, exp in tests:
            text = inp if isinstance(inp, str) else ("\n".join(str(x) for x in inp)
                                                     if isinstance(inp, list) else str(inp))
            sin, sout = io.StringIO(text), io.StringIO()
            ns = {"__name__": "__main__"}
            old_in, old_out, old_err = sys.stdin, sys.stdout, sys.stderr
            try:
                _DL.start(per_case)
                sys.stdin, sys.stdout, sys.stderr = sin, sout, buf
                exec(compile(code, "<sol>", "exec"), ns)
                _DL.stop()
                sys.stdin, sys.stdout, sys.stderr = old_in, old_out, old_err
            except (KeyboardInterrupt, _TLE):
                _DL.stop()
                sys.stdin, sys.stdout, sys.stderr = old_in, old_out, old_err
                case_pass.append(False)
                if first is None:
                    first = {"type": "tle", "err": "case timed out", "args": repr(text)[:200],
                             "expected": repr(exp)[:200], "got": ""}
                continue
            except BaseException as e:
                _DL.stop()
                sys.stdin, sys.stdout, sys.stderr = old_in, old_out, old_err
                case_pass.append(False)
                if first is None:
                    first = {"type": "runtime_error", "err": (type(e).__name__ + ": " + str(e))[:200],
                             "args": repr(text)[:200], "expected": repr(exp)[:200], "got": ""}
                continue
            got = sout.getvalue()
            ok = _out_eq(got, exp)
            case_pass.append(ok)
            if not ok and first is None:
                first = {"type": "wrong_answer", "err": "", "args": repr(text)[:200],
                         "expected": repr(exp)[:200], "got": repr(got.strip())[:200]}
    sys.stdout.write(json.dumps({"case_pass": case_pass, "stage": stage, "first": first}))
    sys.stdout.flush(); os._exit(0)

main()
"""
TACO_RUNNER_PY = os.path.join(RUNNER_DIR, "taco_grader.py")
open(TACO_RUNNER_PY, "w").write(_TACO_RUNNER)
print("runner written to", TACO_RUNNER_PY)

In [ ]:
# ── 2.2 · Executor + self-tests ───────────────────────────────────────────────
import ast, tempfile, subprocess, sys

MAX_CODE_CHARS = 100_000
def syntax_ok(c):
    if not c or not c.strip(): return False, "empty"
    if len(c) > MAX_CODE_CHARS: return False, "degenerate generation"
    try:
        ast.parse(c); return True, ""
    except SyntaxError as e:
        return False, f"{e.msg} (line {e.lineno})"
    except Exception:
        return False, "unparseable"

def run_code(prob, code_str, timeout=None):
    """Grade a candidate on ALL of prob's cases.
    -> (ok, case_pass list, err_type, detail). Gate uses case_pass[:PUBLIC_GATE_K]."""
    timeout = timeout or EXEC_TIMEOUT
    n = len(prob["inputs"])
    ok, msg = syntax_ok(code_str)
    if not ok:
        return False, [False] * n, "syntax_error", {"type": "syntax_error", "err": msg}
    payload = {"code": code_str, "mode": "fn" if prob.get("fn_name") else "stdin",
               "fn_name": prob.get("fn_name"), "per_case_timeout": PER_CASE_TIMEOUT,
               "tests": [[i, o] for i, o in zip(prob["inputs"], prob["outputs"])]}
    with tempfile.NamedTemporaryFile("w", suffix=".json", delete=False) as f:
        json.dump(payload, f); pth = f.name
    try:
        pr = subprocess.run([sys.executable, TACO_RUNNER_PY, pth],
                            capture_output=True, text=True, timeout=timeout)
        if not pr.stdout.strip():
            return False, [False] * n, "runtime_error", {
                "type": "runtime_error", "err": (pr.stderr or "no output")[-200:]}
        d = json.loads(pr.stdout.strip())
        cp = [bool(x) for x in d["case_pass"]]
        cp += [False] * (n - len(cp))
        det = d.get("first") or {"type": "", "err": ""}
        et = "" if all(cp) and cp else det.get("type", "wrong_answer")
        return (bool(cp) and all(cp)), cp, et, det
    except subprocess.TimeoutExpired:
        return False, [False] * n, "tle", {"type": "tle", "err": f"candidate exceeded {timeout}s"}
    except Exception as e:
        return False, [False] * n, "runtime_error", {"type": "runtime_error", "err": str(e)[:200]}
    finally:
        try: os.unlink(pth)
        except OSError: pass

def gate_pass(case_pass):
    """Cascade escalation rule: ship if the first PUBLIC_GATE_K cases pass."""
    k = min(PUBLIC_GATE_K, len(case_pass))
    return k > 0 and all(case_pass[:k])

# ---- self-tests: both problem shapes ----------------------------------------
_std = {"inputs": ["1 2", "10 20", "0 5"], "outputs": ["3", "30", "5"]}
_fn  = {"inputs": [[1, 2], [10, 20]], "outputs": [3, 30], "fn_name": "add"}
_fnw = {"inputs": [[1, 2]], "outputs": [[3]], "fn_name": "add"}          # wrapped expectation
_lst = {"inputs": [[[3, 1, 2]]], "outputs": [[[1, 2, 3]]], "fn_name": "f"}

assert run_code(_std, "a,b=map(int,input().split());print(a+b)")[0]
assert run_code(_std, "import sys\nif __name__=='__main__':\n    a,b=map(int,input().split());print(a+b)")[0]
_ok, _cp, _et, _det = run_code(_std, "a,b=map(int,input().split());print(a-b)")
assert not _ok and _et == "wrong_answer" and _det["expected"] == "'3'" and "-1" in _det["got"]
assert run_code(_std, "print('3')")[1] == [True, False, False]           # per-case vector works
assert gate_pass([True, True, False]) and not gate_pass([True, False, True])
assert run_code(_std, "while True: pass")[2] == "tle"
assert run_code(_fn, "def add(a,b):\n    return a+b\n")[0]
assert run_code(_fnw, "def add(a,b):\n    return 3\n")[0]                # [3] vs 3 both accepted
assert run_code(_lst, "def f(a):\n    return sorted(a)\n")[0]
assert run_code(_fn, "def other(a,b):\n    return a+b\n")[2] == "missing_fn"
assert run_code(_fn, "def add(a,b)\n    return a+b\n")[2] == "syntax_error"
assert run_code(_fn, "def add(a,b):\n    return a+b\nprint('chatter')\n")[0]   # stdout ignored
print("executor OK: stdin + call-based, main-guards, wrapped expectations, per-case vector, "
      "timeouts, failure detail")

## 3 · Generation

Two backends behind one interface. **vLLM** batches every problem in a single call per (model,
temperature) — that is what makes 13 models × ~450 problems × 11 generations affordable. **HF
transformers** is the fallback.

Prompting is shape-aware and model-aware: stdin problems get a "read from stdin, print to
stdout" system prompt, call-based problems get "write exactly this function", and **base**
models get a completion-style prompt with stop sequences instead of a chat template.

In [ ]:
# ── 3.1 · Prompting + code extraction ─────────────────────────────────────────
SYSTEM_STDIN = ("You are an expert competitive programmer. Write a correct Python 3 solution "
                "that reads input from stdin with input() and prints the answer to stdout with "
                "print(). Output ONLY raw Python code -- no markdown, no explanations.")
SYSTEM_FN    = ("You are an expert Python programmer. Write a correct Python 3 function with "
                "exactly the name and signature requested. Output ONLY the raw function "
                "definition -- no markdown, no test code, no main block.")
# v4: StarCoder2 was pretrained on repo-level data joined by <file_sep>. Decoding with
# skip_special_tokens=True deletes that token and leaves a bare path line, after which the
# model happily writes the NEXT file of an imaginary repo. 17.7% of its greedy generations
# were contaminated this way. The literal strings are kept as a belt-and-braces stop.
STOP_BASE = ["\n# Test", "\nif __name__", "\nassert ", "\nprint(input", "\n\n\n\n",
             "<file_sep>", "<|endoftext|>", "<|file_separator|>"]

_FENCE = re.compile(r"```(?:python|py)?[ \t]*\r?\n(.*?)```", re.DOTALL | re.IGNORECASE)
_CODE_START = re.compile(r"^\s*(import |from |def |class |if |for |while |try:|with |@|"
                         r"[A-Za-z_][A-Za-z0-9_]*\s*[=,]|print\(|input\()")

# v4: CodeLlama-Instruct wraps answers in [PYTHON]...[/PYTHON], not markdown fences. v3's
# extract_code knew only about fences, so the closing tag stayed in the saved code and broke
# ast.parse. 123/256 CodeLlama-13B greedy generations were failed for this reason alone
# (syntax-validity 49.6% -> 88.7% once stripped). Same family of artefact: Mistral's [INST].
_MODEL_TAG = re.compile(r"\[/?(?:PYTHON|INST|SYS|TESTS?|CODE|SOL(?:UTION)?)\]", re.IGNORECASE)

# A bare absolute path on its own line = the <file_sep> artefact described above.
_PATHLINE = re.compile(r"^[ \t]*/[\w./-]+\.(?:py|txt|md|java|cpp|c|js|json|html|sh)[ \t]*$",
                       re.MULTILINE)

def strip_model_tags(raw):
    return _MODEL_TAG.sub("", raw or "")

def cut_at_next_file(raw):
    """Truncate at the first stray file-path line (base-model repo continuation)."""
    m = _PATHLINE.search(raw or "")
    return raw[:m.start()] if m else raw

def extract_code(raw):
    raw = strip_model_tags(raw).strip()
    raw = cut_at_next_file(raw)
    blocks = _FENCE.findall(raw)
    if blocks:
        return max(blocks, key=len).strip()
    if "```" in raw:
        cand = max(raw.split("```"), key=len)
        raw = re.sub(r"^(python|py)\b", "", cand.strip(), flags=re.IGNORECASE).strip()
    lines = raw.split("\n")
    for i, ln in enumerate(lines):
        if _CODE_START.match(ln):
            raw = "\n".join(lines[i:]); break
    return raw.strip("`").strip()

def truncate_at_stop(text, stops):
    cut = len(text)
    for s in stops:
        i = text.find(s)
        if i != -1: cut = min(cut, i)
    return text[:cut]

def task_text(prob):
    fn = prob.get("fn_name")
    if fn:
        t = f"Write a Python function named `{fn}`.\n\n{prob['question']}"
        if prob.get("starter"): t += f"\n\nUse this signature:\n{prob['starter']}"
        if prob["inputs"]:
            t += f"\n\nExample: {fn}(*{prob['inputs'][0]!r}) should return {prob['outputs'][0]!r}"
    else:
        t = prob["question"]
        if prob.get("starter"): t += f"\n\nStarter code:\n{prob['starter']}"
        if prob["inputs"]:
            t += (f"\n\nExample input:\n{prob['inputs'][0]}\n"
                  f"Expected output:\n{prob['outputs'][0]}")
    return t

def build_prompt(tok, cfg, prob):
    sysmsg = SYSTEM_FN if prob.get("fn_name") else SYSTEM_STDIN
    body = task_text(prob)
    if cfg["base"]:
        return f'"""\n{body}\n"""\n', "completion"
    tmpl = getattr(tok, "chat_template", None)
    if tmpl:
        try:
            return tok.apply_chat_template(
                [{"role": "user", "content": sysmsg + "\n\n" + body}],
                tokenize=False, add_generation_prompt=True), "chat"
        except Exception:
            pass
    return f"{sysmsg}\n\n{body}\n\n", "chat"

def postprocess(mode, prob, raw):
    if mode == "completion":
        raw = cut_at_next_file(strip_model_tags(raw or ""))
        return truncate_at_stop(raw, STOP_BASE).strip()
    return extract_code(raw)

assert extract_code("Here:\n```python\nprint(1)\n```\ndone") == "print(1)"
assert truncate_at_stop("print(1)\nif __name__ == '__main__':\n    pass", STOP_BASE) == "print(1)"
# v4 regression tests, both taken verbatim from real v3 output
assert extract_code('print("YES")\n[/PYTHON]') == 'print("YES")'
assert extract_code('[PYTHON]\ndef f():\n    return 1\n[/PYTHON]') == "def f():\n    return 1"
assert postprocess("completion", {}, "main()\n/codeforces/1005/A.py\nimport os\n") == "main()"
import ast as _ast
_ast.parse(extract_code('print("YES") if x else print("NO")\n[/PYTHON]'))
print("prompt/extraction helpers OK (v4: [/PYTHON] + <file_sep> regression tests pass)")

In [ ]:
# ── 3.2 · Backends ────────────────────────────────────────────────────────────
import gc, time, zlib

if BACKEND == "auto":
    BACKEND = "vllm" if (HAS_VLLM and HAS_TORCH and torch.cuda.is_available() and not DRY_RUN) else "hf"
print(f"generation backend: {BACKEND}")

def problem_seed(key): return GLOBAL_SEED + zlib.crc32(key.encode()) % (2**31)

class MockEngine:
    """DRY_RUN: deterministic mock so the pipeline can be smoke-tested without a GPU."""
    progress = None
    def __init__(self, cfg): self.cfg = cfg
    def generate(self, items, n, temp):
        import random
        out = []
        for key, p in items:
            rnd = random.Random(problem_seed(key) + int(temp * 100) + hash(self.cfg["name"]) % 997)
            skill = 0.25 + 0.30 * ((hash(self.cfg["name"] + p["category"]) % 100) / 100.0)
            good = (f"def {p['fn_name']}(*a):\n    return sum(a)\n" if p.get("fn_name")
                    else "a,b=map(int,input().split());print(a+b)")
            bad = ("def _w(*a):\n    return 0\n" if p.get("fn_name") else "print('nope')")
            codes = [good if rnd.random() < skill else bad for _ in range(n)]
            out.append({"codes": codes, "prompt_tokens": 220,
                        "completion_tokens": 90 * n, "time": 0.01 * n})
        if self.progress: self.progress(len(items))
        return out
    def close(self): pass

class VLLMEngine:
    progress = None
    def __init__(self, cfg):
        from vllm import LLM
        from transformers import AutoTokenizer
        self.cfg = cfg
        self.tok = AutoTokenizer.from_pretrained(cfg["repo"], token=HF_TOKEN or None,
                                                 trust_remote_code=True)
        self.llm = LLM(model=cfg["repo"], dtype="bfloat16", trust_remote_code=True,
                       gpu_memory_utilization=GPU_MEM_UTIL, max_model_len=MAX_MODEL_LEN,
                       seed=GLOBAL_SEED, download_dir=os.environ.get("HF_HOME"), swap_space=4)
    def generate(self, items, n, temp):
        from vllm import SamplingParams
        prompts, modes = [], []
        for _, p in items:
            txt, mode = build_prompt(self.tok, self.cfg, p)
            prompts.append(txt); modes.append(mode)
        sp = SamplingParams(n=n, temperature=temp, top_p=(TOP_P if temp > 0 else 1.0),
                            max_tokens=MAX_NEW_TOKENS, seed=GLOBAL_SEED,
                            stop=(STOP_BASE if self.cfg["base"] else None))
        t0 = time.time(); outs = self.llm.generate(prompts, sp); dt = time.time() - t0
        res = []
        for (key, p), mode, o in zip(items, modes, outs):
            res.append({"codes": [postprocess(mode, p, c.text) for c in o.outputs],
                        "prompt_tokens": len(o.prompt_token_ids),
                        "completion_tokens": sum(len(c.token_ids) for c in o.outputs),
                        "time": dt / max(len(items), 1)})
        if self.progress: self.progress(len(items))
        return res
    def close(self):
        try:
            from vllm.distributed.parallel_state import (destroy_model_parallel,
                                                         destroy_distributed_environment)
            destroy_model_parallel(); destroy_distributed_environment()
        except Exception: pass
        for attr in ("llm",):
            try: delattr(self, attr)
            except Exception: pass
        gc.collect()
        if HAS_TORCH and torch.cuda.is_available(): torch.cuda.empty_cache()

class FatalCudaError(RuntimeError):
    """Unrecoverable CUDA context failure -- the kernel must be restarted."""


class HFEngine:
    """Batched HF generation. The old version ran one problem at a time, which leaves a big GPU
    ~90% idle; this batches across problems (length-sorted to minimise padding) and multiplies
    the batch by num_return_sequences for the sampling pass."""
    progress = None
    def __init__(self, cfg):
        from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
        self.cfg = cfg
        four_bit = cfg["in4bit"] and PRECISION_MODE == "as_listed"
        # Prefer the transformers-native implementation. Vendor-bundled remote code is often
        # pinned to an older transformers API -- Phi-3.5's copy calls DynamicCache.get_max_length(),
        # which no longer exists, and the model dies on the first generate(). Only fall back to
        # remote code for architectures transformers genuinely does not know.
        self.trust_remote = bool(cfg.get("trust_remote_code", False))
        try:
            self.tok = AutoTokenizer.from_pretrained(cfg["repo"], token=HF_TOKEN or None,
                                                     trust_remote_code=self.trust_remote)
        except Exception as e:
            # v4: an auth failure is NOT a remote-code problem. v3 caught the 403 on
            # google/codegemma-7b-it, printed "tokenizer needs remote code", retried, hit the
            # same 403, and buried the real cause under two tracebacks.
            _s = str(e)
            if "403" in _s or "401" in _s or "gated repo" in _s.lower() \
                    or "GatedRepo" in type(e).__name__:
                raise RuntimeError(
                    f"{cfg['name']}: {cfg['repo']} is GATED and this HF_TOKEN is not "
                    f"authorised for it. Accept the licence at "
                    f"https://huggingface.co/{cfg['repo']} using the SAME account the token "
                    f"belongs to, then re-run. This is NOT a trust_remote_code problem.") from e
            if self.trust_remote: raise
            print(f"    tokenizer needs remote code ({type(e).__name__}); retrying with it")
            self.trust_remote = True
            self.tok = AutoTokenizer.from_pretrained(cfg["repo"], token=HF_TOKEN or None,
                                                     trust_remote_code=True)
        if self.tok.pad_token is None: self.tok.pad_token = self.tok.eos_token
        self.tok.padding_side = "left"        # required for correct batched decoder generation
        bnb = (BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
                                  bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4")
               if four_bit else None)
        kw = dict(token=HF_TOKEN or None, device_map="auto",
                  torch_dtype=None if four_bit else torch.bfloat16, quantization_config=bnb,
                  trust_remote_code=self.trust_remote)
        try:
            self.model = AutoModelForCausalLM.from_pretrained(cfg["repo"],
                             attn_implementation=ATTN_IMPL, **kw)
            self.attn = ATTN_IMPL
        except Exception as e:                # some architectures lack sdpa/flash kernels
            print(f"    {ATTN_IMPL} unavailable ({type(e).__name__}); falling back to eager")
            self.model = AutoModelForCausalLM.from_pretrained(cfg["repo"],
                             attn_implementation="eager", **kw)
            self.attn = "eager"
        self.model.eval()
        self.model.generation_config.use_cache = True
        # Chat models often end a turn with a template token that is NOT eos_token. If it is not
        # registered, every sample runs to max_new_tokens: 4x slower, 4x the KV cache (this is
        # what OOMed DeepSeek), and trailing garbage after the function.
        self.eos_ids = []
        if self.tok.eos_token_id is not None:
            self.eos_ids.append(self.tok.eos_token_id)
        for t in ["<|EOT|>", "<|im_end|>", "<|eot_id|>", "<end_of_turn>",
                  "<|end|>", "<|endoftext|>", "<|end_of_text|>", "<|end▁of▁sentence|>"]:
            tid = self.tok.convert_tokens_to_ids(t)
            if isinstance(tid, int) and tid >= 0 and tid != self.tok.unk_token_id \
                    and tid not in self.eos_ids:
                self.eos_ids.append(tid)
        self.precision = "nf4" if four_bit else "bf16"
        # Hard-cap this process's share of the card. On Windows the driver otherwise falls
        # back to system RAM instead of raising OOM: no error, ~70x slower, and the notebook
        # cannot react to it. With the cap, over-allocation becomes a normal recoverable
        # torch.cuda.OutOfMemoryError that the batch backoff below handles.
        if torch.cuda.is_available():
            try:
                torch.cuda.set_per_process_memory_fraction(GPU_MEM_FRACTION)
            except Exception as e:
                print(f"    (could not set memory fraction: {e})")

        self._n_cap = N_SAMPLES      # learned ceiling on num_return_sequences per call
        self._last_bs = HF_MAX_SEQS  # learned prompt-batch size, persists across chunks
        per_seq = self._per_seq_gb(PROMPT_TOKEN_BUDGET + MAX_NEW_TOKENS)
        free = (torch.cuda.mem_get_info()[0] / 1024**3) if torch.cuda.is_available() else 24.0
        fits = int((KV_SAFETY_FRAC * free) / max(per_seq, 1e-6))
        self.bs = max(1, min(HF_MAX_SEQS, fits))
        print(f"    attn={self.attn} | {self.precision} | eos_ids={self.eos_ids}")
        print(f"    VRAM free {free:.1f} GB | ~{per_seq*1000:.0f} MB/seq @{PROMPT_TOKEN_BUDGET}"
              f"+{MAX_NEW_TOKENS} tok | max_seqs={self.bs} (ceiling {HF_MAX_SEQS}, fits {fits})")
        _worst = self._per_seq_gb(MAX_MODEL_LEN)
        print(f"    worst-case prompt ({MAX_MODEL_LEN} tok): ~{_worst*1000:.0f} MB/seq -> "
              f"{max(1, int(KV_SAFETY_FRAC * free / max(_worst,1e-6)))} concurrent seqs")
        if self.bs < N_SAMPLES:
            print(f"    NOTE: {N_SAMPLES} samples will be split into "
                  f"{-(-N_SAMPLES // self.bs)} calls of <= {self.bs} sequences.")
        if fits < 1:
            print("    WARNING: not enough free VRAM for even one sequence at this length. "
                  "Close other GPU processes or lower MAX_NEW_TOKENS.")

    def _per_seq_gb(self, ctx_tokens):
        """bf16 KV cache for one sequence at this context length."""
        c = self.model.config
        try:
            nl = c.num_hidden_layers
            nkv = getattr(c, "num_key_value_heads", None) or c.num_attention_heads
            hd = getattr(c, "head_dim", None) or (c.hidden_size // c.num_attention_heads)
            return 2 * 2 * nl * nkv * hd * ctx_tokens / 1024**3
        except Exception:
            return 0.5

    def _seq_budget(self, ctx_tokens):
        """How many concurrent sequences fit RIGHT NOW at this ACTUAL context length."""
        per = self._per_seq_gb(ctx_tokens)
        free = (torch.cuda.mem_get_info()[0] / 1024**3) if torch.cuda.is_available() else 24.0
        return max(1, min(HF_MAX_SEQS, int((KV_SAFETY_FRAC * free) / max(per, 1e-6)))), per, free

    def _sub_batch(self, texts, modes, probs, idxs, n, temp, tag):
        tok, mdl = self.tok, self.model
        # pad_to_multiple_of collapses many distinct prompt lengths into a handful of shapes.
        # Without it every sub-batch allocates a differently-shaped block, the caching allocator
        # cannot reuse the old ones, and reserved memory ratchets up until the card is full.
        enc = tok([texts[j] for j in idxs], return_tensors="pt", padding=True,
                  pad_to_multiple_of=128, truncation=True,
                  max_length=MAX_MODEL_LEN - MAX_NEW_TOKENS).to(mdl.device)
        il = enc["input_ids"].shape[1]
        kw = dict(max_new_tokens=MAX_NEW_TOKENS, pad_token_id=tok.pad_token_id, use_cache=True,
                  eos_token_id=self.eos_ids or tok.eos_token_id)
        # temperature/top_p/top_k must be cleared for greedy, or transformers warns on every call
        # because the model's own generation_config presets them
        kw.update(dict(do_sample=True, temperature=temp, top_p=TOP_P) if temp > 0
                  else dict(do_sample=False, temperature=None, top_p=None, top_k=None))
        if n > 1: kw["num_return_sequences"] = n
        if self.cfg["base"]:
            kw.update(stop_strings=STOP_BASE, tokenizer=tok)   # transformers >= 4.39
        torch.manual_seed(GLOBAL_SEED + tag)
        t0 = time.time()
        try:
            with torch.no_grad(): outs = mdl.generate(**enc, **kw)
        except TypeError:                     # older transformers: no stop_strings support
            kw.pop("stop_strings", None); kw.pop("tokenizer", None)
            with torch.no_grad(): outs = mdl.generate(**enc, **kw)
        dt = (time.time() - t0) / max(len(idxs), 1)
        gen = outs[:, il:]
        out = {}
        for bi, j in enumerate(idxs):
            seqs = gen[bi * n:(bi + 1) * n]
            codes = [postprocess(modes[j], probs[j],
                                 tok.decode(s, skip_special_tokens=True)) for s in seqs]
            out[j] = {"codes": codes,
                      "prompt_tokens": int(enc["attention_mask"][bi].sum()),
                      "completion_tokens": int((seqs != tok.pad_token_id).sum()),
                      "time": dt}
        # Drop references to the KV cache and output tensors before the next sub-batch, then
        # hand the freed blocks back. Costs a few ms; without it reserved memory only grows.
        del outs, gen, enc
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return out

    def _generate(self, probs, n, temp):
        """v4 rewrite. Two changes from v3:

        1. The batch is sized from the ACTUAL longest prompt in the slice, not from the fixed
           PROMPT_TOKEN_BUDGET guess. Prompts truncate at 3328 tokens, so the old estimate
           under-counted the KV cache by up to 2.4x on long TACO statements.
        2. The OOM ladder has a third rung. v3 could only halve the PROMPT count; once it hit
           1 prompt x n sequences it gave up and wrote empty strings. On a 13B in bf16 that is
           exactly where it landed (self.bs=8, n=10 -> bs = 8//10 -> 1 prompt x 10 seqs), so
           43/256 CodeLlama-13B problems were checkpointed as permanent zeros. Now n itself is
           split: 10 samples become 2 calls of 5, then 5 of 2, then 10 of 1.
        """
        texts, modes = [], []
        for p in probs:
            t, m = build_prompt(self.tok, self.cfg, p)
            texts.append(t); modes.append(m)
        lens = [len(self.tok(t).input_ids) for t in texts]
        order = sorted(range(len(probs)), key=lambda i: lens[i])   # length-sorted = less padding
        results = [None] * len(probs)

        i = 0
        while i < len(order):
            # look at the prompts we are about to send, and budget for the LONGEST of them
            probe = order[i:i + max(self._last_bs, 1)]
            ctx = min(MAX_MODEL_LEN, max(lens[j] for j in probe) + MAX_NEW_TOKENS)
            budget, per, free = self._seq_budget(ctx)
            n_chunk = max(1, min(n, budget, self._n_cap))
            bs = max(1, min(budget // n_chunk, len(order) - i))
            idxs = order[i:i + bs]

            acc = {j: {"codes": [], "prompt_tokens": lens[j],
                       "completion_tokens": 0, "time": 0.0} for j in idxs}
            got, failed = 0, False
            while got < n:
                take = min(n_chunk, n - got)
                try:
                    part = self._sub_batch(texts, modes, probs, idxs, take, temp,
                                           i + got + int(temp * 1000))
                    for j, r in part.items():
                        acc[j]["codes"] += r["codes"]
                        acc[j]["completion_tokens"] += r["completion_tokens"]
                        acc[j]["time"] += r["time"]
                        acc[j]["prompt_tokens"] = r["prompt_tokens"]
                    got += take
                except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
                    # A driver-level "CUDA error: out of memory" is a RuntimeError, NOT
                    # torch.cuda.OutOfMemoryError, and it poisons the CUDA context: every later
                    # call (including empty_cache) raises. Only allocator OOM is recoverable.
                    msg = str(e).lower()
                    if "out of memory" not in msg:
                        raise
                    if "cuda error" in msg:
                        raise FatalCudaError(
                            "driver-level CUDA OOM -- the CUDA context cannot be recovered in "
                            "this process. Progress is checkpointed; restart the kernel, lower "
                            f"HF_MAX_SEQS (currently {self.bs}), and re-run to resume.") from e
                    try:
                        torch.cuda.empty_cache(); gc.collect()
                    except Exception:
                        pass
                    if bs > 1:                       # rung 1: fewer prompts
                        bs = max(1, bs // 2)
                        idxs = order[i:i + bs]
                        acc = {j: {"codes": [], "prompt_tokens": lens[j],
                                   "completion_tokens": 0, "time": 0.0} for j in idxs}
                        got = 0
                        print(f"    OOM -> {bs} prompts x {n_chunk} seqs")
                    elif n_chunk > 1:                # rung 2: fewer samples per call  [NEW]
                        n_chunk = max(1, n_chunk // 2)
                        self._n_cap = n_chunk
                        acc = {j: {"codes": [], "prompt_tokens": lens[j],
                                   "completion_tokens": 0, "time": 0.0} for j in idxs}
                        got = 0
                        print(f"    OOM -> splitting samples: 1 prompt x {n_chunk} seqs "
                              f"({n} total in {-(-n // n_chunk)} calls)")
                    else:                            # rung 3: one sequence will not fit
                        print(f"    OOM at 1 prompt x 1 seq (ctx {ctx} tok, ~{per*1000:.0f} "
                              f"MB/seq, {free:.1f} GB free) -- SKIPPING these {len(idxs)} "
                              f"problem(s); they stay in the todo list for the next run")
                        failed = True
                        break
            self._last_bs = bs
            if not failed:
                for j in idxs:
                    results[j] = acc[j]              # results[j] stays None on failure
            i += len(idxs)
            if self.progress: self.progress(len(idxs))
            if torch.cuda.is_available() and (i // max(bs, 1)) % 5 == 0:
                res = torch.cuda.memory_reserved() / 1024**3
                alloc = torch.cuda.memory_allocated() / 1024**3
                if res > 0.85 * (torch.cuda.get_device_properties(0).total_memory / 1024**3):
                    print(f"    NOTE: reserved {res:.1f} GB (allocated {alloc:.1f}) -- close to "
                          f"the card limit; lower HF_MAX_SEQS if throughput drops")
        return results

    def close(self):
        try: del self.model, self.tok
        except Exception: pass
        gc.collect()
        try:                      # after a fatal CUDA error even this raises -- never mask
            if HAS_TORCH and torch.cuda.is_available(): torch.cuda.empty_cache()
        except Exception as e:
            print(f"    (empty_cache failed: {type(e).__name__} -- restart the kernel)")
    def generate(self, items, n, temp):
        return self._generate([p for _, p in items], n, temp)

def make_engine(cfg):
    if DRY_RUN: return MockEngine(cfg)
    return VLLMEngine(cfg) if BACKEND == "vllm" else HFEngine(cfg)

def gpu_mem_str():
    if not (HAS_TORCH and torch.cuda.is_available()): return "cpu"
    free, total = torch.cuda.mem_get_info()
    return f"{(total-free)/1024**3:.1f}/{total/1024**3:.1f} GB"
print("engines ready |", gpu_mem_str())

## 4 · Generation and grading sweeps (crash-safe)

Generations append to `gen/<model>.jsonl`, one `fsync`ed line per problem. A `kill -9` mid-write
loses only the final partial line, which the loader discards — a rewritten `.json` would lose
everything. Re-running regenerates only what is missing.

Generation and grading are deliberately **separate stages**. Change a timeout, fix the output
comparison, adjust `PUBLIC_GATE_K` — you re-grade in minutes without spending a GPU-hour again.

In [ ]:
# ── 4.1 · JSONL checkpoint helpers ────────────────────────────────────────────
def jsonl_path(stage, model):
    return os.path.join(RESULTS_DIR, stage, f"{re.sub(r'[^A-Za-z0-9]+','_',model)}.jsonl")

def jsonl_load(path, key="key"):
    out = {}
    if not os.path.exists(path): return out
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line: continue
            try: rec = json.loads(line)
            except json.JSONDecodeError: continue     # truncated last line after a crash
            out[rec[key]] = rec
    return out

def jsonl_append(path, rec):
    with open(path, "a") as f:
        f.write(json.dumps(rec) + "\n"); f.flush(); os.fsync(f.fileno())
print("checkpoint helpers ready")

### 4.1b · Maintenance utilities (v4)

`audit_generations()` reports empty / tag-contaminated records in every checkpoint. `scrub_empty_generations(apply=True)` removes unusable records so the sweep regenerates them (a `.bak` is kept). `reprocess_generations(apply=True)` re-runs the **fixed** `extract_code` over text already on disk — it repairs the `[/PYTHON]` and `<file_sep>` damage with no GPU time at all.


In [ ]:
# ── 4.1b · Maintenance utilities (v4) ────────────────────────────────────────
import ast as _ast

def _syntax_ok(code):
    if not (code or "").strip(): return False
    try: _ast.parse(code); return True
    except Exception: return False

def _read_jsonl(path):
    recs = []
    if not os.path.exists(path): return recs
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            try: recs.append(json.loads(line))
            except json.JSONDecodeError: continue
    return recs

def _unusable(r):
    """No greedy text, or every sample empty, or short sample list."""
    sc = r.get("sample_codes") or []
    return (not (r.get("greedy_code") or "").strip()
            or not sc
            or len(sc) < N_SAMPLES
            or all(not (c or "").strip() for c in sc))

def audit_generations(models=None):
    """Read-only health report over the gen checkpoints."""
    models = models or [m["name"] for m in MODELS]
    print(f"{'model':<22} {'recs':>5} {'unusable':>9} {'gSyntaxOK':>10} {'sSyntaxOK':>10} {'tags':>6}")
    for name in models:
        recs = _read_jsonl(jsonl_path("gen", name))
        if not recs: continue
        bad = sum(1 for r in recs if _unusable(r))
        gok = sum(1 for r in recs if _syntax_ok(r.get("greedy_code")))
        st = [c for r in recs for c in (r.get("sample_codes") or [])]
        sok = sum(1 for c in st if _syntax_ok(c))
        tag = sum(1 for r in recs
                  if _MODEL_TAG.search(r.get("greedy_code") or "")
                  or _PATHLINE.search(r.get("greedy_code") or ""))
        print(f"{name:<22} {len(recs):5d} {bad:9d} {gok/len(recs):9.1%} "
              f"{(sok/len(st) if st else 0):9.1%} {tag:6d}")

def scrub_empty_generations(models=None, apply=False):
    """Drop unusable records so the generation sweep retries them. Keeps a .bak."""
    models = models or [m["name"] for m in MODELS]
    for name in models:
        path = jsonl_path("gen", name)
        if not os.path.exists(path): continue
        keep, drop = [], 0
        with open(path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line: continue
                try: r = json.loads(line)
                except json.JSONDecodeError: continue
                if _unusable(r): drop += 1
                else: keep.append(line)
        if drop:
            print(f"  {name:<22} {drop} unusable of {drop+len(keep)}"
                  f"{' -> removed' if apply else '  (dry run)'}")
            if apply:
                os.replace(path, path + ".bak")
                with open(path, "w", encoding="utf-8") as f:
                    f.write("\n".join(keep) + ("\n" if keep else ""))
        else:
            print(f"  {name:<22} clean ({len(keep)} records)")

def reprocess_generations(models=None, apply=False):
    """Re-run the FIXED extract_code over saved text. No GPU needed.

    Only useful for checkpoints written by v3, where [/PYTHON] and the <file_sep> path
    artefact were left in the stored code. Records written by v4 are already clean, so this
    is a no-op on them.
    """
    models = models or [m["name"] for m in MODELS]
    for name in models:
        path = jsonl_path("gen", name)
        recs = _read_jsonl(path)
        if not recs: continue
        base = next((m for m in MODELS if m["name"] == name), None)
        mode = "completion" if (base and base["base"]) else "chat"
        changed = g_before = g_after = 0
        for r in recs:
            g0 = r.get("greedy_code") or ""
            g1 = postprocess(mode, {}, g0)
            s1 = [postprocess(mode, {}, c or "") for c in (r.get("sample_codes") or [])]
            g_before += _syntax_ok(g0); g_after += _syntax_ok(g1)
            if g1 != g0 or s1 != (r.get("sample_codes") or []):
                changed += 1
                r["greedy_code"] = g1; r["sample_codes"] = s1
        n = len(recs)
        print(f"  {name:<22} {changed:4d}/{n} records changed | greedy syntax-ok "
              f"{g_before/n:.1%} -> {g_after/n:.1%}{'' if apply else '  (dry run)'}")
        if apply and changed:
            os.replace(path, path + ".preproc.bak")
            with open(path, "w", encoding="utf-8") as f:
                for r in recs: f.write(json.dumps(r) + "\n")

print("maintenance utilities ready:")
print("  audit_generations()                      # health report, read-only")
print("  reprocess_generations(apply=True)        # re-extract v3 text with the fixed parser")
print("  scrub_empty_generations(apply=True)      # drop unusable records so they regenerate")


In [ ]:
# ── 4.2 · Generation sweep ────────────────────────────────────────────────────
from tqdm.auto import tqdm
import traceback

GEN_CHUNK = 64 if BACKEND == "vllm" else 16   # problems between checkpoint writes. Smaller on
                                              # the HF path so progress and checkpoints are
                                              # visible; batching happens inside the engine.

def generate_for_model(cfg):
    path = jsonl_path("gen", cfg["name"])
    done = jsonl_load(path)
    todo = [(k, PROBS[k]) for k in KEYS_ALL if k not in done]
    print(f"\n=== {cfg['name']}: {len(done)}/{len(KEYS_ALL)} done | VRAM {gpu_mem_str()}")
    if not todo:
        print("    complete, skipping"); return
    eng = make_engine(cfg)
    pbar = tqdm(total=2 * len(todo), desc=cfg["name"], unit="gen-pass",
                postfix="greedy+samples")
    eng.progress = pbar.update          # ticks per sub-batch, not per chunk
    skipped = 0
    try:
        for i in range(0, len(todo), GEN_CHUNK):
            batch = todo[i:i + GEN_CHUNK]
            g = eng.generate(batch, 1, GREEDY_TEMP)
            s = eng.generate(batch, N_SAMPLES, SAMPLE_TEMP)
            for (key, prob), gr, sr in zip(batch, g, s):
                # v4: a record with no usable text is WORSE than no record at all -- it marks
                # the problem done forever, so the resume skips it and the model scores a
                # permanent 0. Leave it in the todo list instead.
                incomplete = (
                    gr is None or sr is None
                    or not gr["codes"] or not (gr["codes"][0] or "").strip()
                    or len(sr["codes"]) < N_SAMPLES
                    or all(not (c or "").strip() for c in sr["codes"]))
                if SKIP_EMPTY_RECORDS and incomplete:
                    skipped += 1
                    continue
                jsonl_append(path, {
                    "key": key, "model": cfg["name"], "category": prob["category"],
                    "difficulty": prob["difficulty"],
                    "problem_type": "fn" if prob.get("fn_name") else "stdin",
                    "greedy_code": gr["codes"][0] if gr["codes"] else "",
                    "sample_codes": sr["codes"],
                    "prompt_tokens": gr["prompt_tokens"],
                    "completion_tokens_greedy": gr["completion_tokens"],
                    "completion_tokens_samples": sr["completion_tokens"],
                    "time_greedy": gr["time"], "time_samples": sr["time"],
                    "precision": getattr(eng, "precision", "bf16"), "backend": BACKEND})
    finally:
        pbar.close()
        if skipped:
            print(f"    !! {skipped} problem(s) NOT written (OOM or empty output).")
            print(f"       They remain in the todo list -- re-run this cell to retry them.")
        try:
            eng.close()
        except Exception as e:
            print(f"    close failed ({type(e).__name__}) -- restart the kernel before retrying")
        try:
            print(f"    unloaded | VRAM {gpu_mem_str()}")
        except Exception:
            pass

FATAL = False
for cfg in [m for m in MODELS if m["enabled"]]:
    try:
        generate_for_model(cfg)
    except KeyboardInterrupt:
        print("interrupted -- progress is on disk, re-run to resume"); raise
    except FatalCudaError as e:
        # The CUDA context is dead: every later model would fail the same way, and cleanup
        # calls raise too. Stop now rather than emit twelve identical tracebacks.
        print(f"\n!! {cfg['name']}: {e}")
        FATAL = True
        break
    except Exception:
        print(f"!! {cfg['name']} failed; progress saved, continuing with the next model")
        traceback.print_exc()
        try:
            if HAS_TORCH and torch.cuda.is_available():
                torch.cuda.empty_cache(); gc.collect()
        except Exception:
            print("    (cleanup failed -- restart the kernel)")
            FATAL = True; break

if FATAL:
    print("\n" + "=" * 72)
    print("STOPPED ON AN UNRECOVERABLE CUDA ERROR.")
    print("Everything generated so far is on disk. To continue:")
    print("  1. Kernel -> Restart")
    print("  2. lower HF_MAX_SEQS in the config cell (halve it)")
    print("  3. re-run from the top -- completed problems are skipped automatically")
    print("=" * 72)
else:
    print("\nGENERATION SWEEP FINISHED")

### If a model OOMs, or vLLM will not release the GPU

Restart the kernel and re-run the notebook. Everything already generated is on disk, so the
sweep resumes exactly where it stopped. If one model keeps failing, set `"enabled": False` for
it — it then drops cleanly out of the analysis rather than leaving a half-finished column.

In [ ]:
# ── 4.3 · Grading sweep (CPU, parallel, resumable) ────────────────────────────
from concurrent.futures import ThreadPoolExecutor, as_completed
# Threads, not processes: each grade_one spends its time inside subprocess.run (which releases
# the GIL), and a thread pool avoids the "cannot pickle a function defined in a notebook cell"
# failure that ProcessPoolExecutor hits on some kernels.

def grade_one(args):
    key, model, greedy_code, sample_codes = args
    prob = PROBS[key]
    g_ok, g_cases, g_et, g_det = run_code(prob, greedy_code)
    sp, se = [], []
    for c in sample_codes:
        ok, _, et, _ = run_code(prob, c)
        sp.append(bool(ok)); se.append(et)
    return {"key": key, "model": model, "category": prob["category"],
            "difficulty": prob["difficulty"],
            "problem_type": "fn" if prob.get("fn_name") else "stdin",
            "greedy_pass": bool(g_ok), "greedy_err": g_et,
            "greedy_cases": g_cases, "greedy_gate": bool(gate_pass(g_cases)),
            "greedy_npass": int(sum(g_cases)), "greedy_ntot": len(g_cases),
            "greedy_detail": {k: g_det.get(k) for k in ("type", "err", "args", "expected", "got")},
            "sample_pass": sp, "sample_errs": se}

def grade_for_model(name):
    gen = jsonl_load(jsonl_path("gen", name))
    gpath = jsonl_path("grade", name)
    done = jsonl_load(gpath)
    todo = [(k, name, gen[k]["greedy_code"], gen[k]["sample_codes"])
            for k in KEYS_ALL if k in gen and k not in done]
    if not todo:
        print(f"  {name:<22} graded {len(done)}/{len(gen)}"); return
    with ThreadPoolExecutor(max_workers=N_EXEC_WORKERS) as ex:
        futs = {ex.submit(grade_one, a): a[0] for a in todo}
        for fu in tqdm(as_completed(futs), total=len(futs), desc=f"grade:{name}", unit="prob"):
            try:
                jsonl_append(gpath, fu.result())
            except Exception as e:
                print(f"    grading crashed on {futs[fu]}: {e}")
    print(f"  {name:<22} graded {len(jsonl_load(gpath))}/{len(gen)}")

for name in ENABLED:
    grade_for_model(name)
print("\nGRADING FINISHED")

## 5 · Load results, coverage guard, headline tables

Analysis runs on the intersection of problems graded for every model, so every comparison is
paired on identical problems. Partial coverage and smoke-test runs print a loud banner — those
numbers are not results.

In [ ]:
# ── 5.1 · Load + coverage guard ───────────────────────────────────────────────
import numpy as np, pandas as pd, math
from math import comb
from collections import defaultdict, Counter

RAW, GEN = {}, {}
for name in ENABLED:
    g = jsonl_load(jsonl_path("grade", name))
    if g:
        RAW[name] = g
        GEN[name] = jsonl_load(jsonl_path("gen", name))
missing = [m for m in ENABLED if m not in RAW]
partial = {m: len(KEYS_ALL) - len(RAW[m]) for m in RAW if len(RAW[m]) < len(KEYS_ALL)}
if missing: print(f"not yet evaluated : {missing}")
if partial: print(f"incomplete (remaining problems): {partial}")

keys = sorted(set.intersection(*[set(RAW[m]) for m in RAW])) if RAW else []
if not keys:
    raise SystemExit("No graded results yet. Run the generation (4.2) and grading (4.3) cells "
                     "first, or set DRY_RUN=True for a no-GPU pipeline smoke test.")
MAIN = [m for m in RAW if not IS_BASE[m]]
BASE = [m for m in RAW if IS_BASE[m]]
ref0 = list(RAW)[0]
cat_of  = {k: RAW[ref0][k]["category"] for k in keys}
diff_of = {k: RAW[ref0][k]["difficulty"] for k in keys}
type_of = {k: RAW[ref0][k]["problem_type"] for k in keys}
cats  = sorted(set(cat_of.values()))
cat_n = Counter(cat_of.values())
order = sorted(cats, key=lambda c: -cat_n[c])

PARTIAL = bool(missing or partial or len(keys) < len(KEYS_ALL))
NOT_RESULTS = TRIAL_MODE or QUICK_TEST or DRY_RUN
if NOT_RESULTS:
    print("\n" + "!" * 72 +
          "\n  TRIAL / QUICK / DRY DATA -- EVERYTHING BELOW IS A SMOKE TEST, NOT RESULTS\n" + "!" * 72)
if PARTIAL:
    print("\nPARTIAL COVERAGE -- numbers below are not final results.")
print(f"\nanalysis set: {len(keys)} problems x {len(RAW)} models "
      f"({len(MAIN)} in the routing pool, {len(BASE)} base models benchmarked but excluded)")
print(f"difficulty : {dict(Counter(diff_of.values()))}")
print(f"types      : {dict(Counter(type_of.values()))}")

In [ ]:
# ── 5.2 · Tidy frame, unbiased pass@k, cost model ─────────────────────────────
def upass(c, n, k):
    """Unbiased pass@k (Chen et al., 2021)."""
    if n == 0 or n < k: return float("nan")
    if n - c < k: return 1.0
    return 1.0 - comb(n - c, k) / comb(n, k)

def wilson(p, n, z=1.96):
    if n == 0 or p != p: return float("nan"), float("nan")
    d = 1 + z * z / n
    ctr = (p + z * z / (2 * n)) / d
    hw = z * math.sqrt(max(p * (1 - p) / n + z * z / (4 * n * n), 0)) / d
    return ctr - hw, ctr + hw

rows = []
for m in RAW:
    price = PRICE[m]
    for k in keys:
        rec, gen = RAW[m][k], GEN[m][k]
        sp = rec.get("sample_pass") or []
        c, n = int(sum(sp)), len(sp)
        ptok = gen.get("prompt_tokens", 0)
        gtok = gen.get("completion_tokens_greedy", 0)
        stok_mean = (gen.get("completion_tokens_samples", 0) / n) if n else float("nan")
        row = {"model": m, "key": k, "category": rec["category"], "difficulty": rec["difficulty"],
               "problem_type": rec["problem_type"], "greedy": int(rec["greedy_pass"]),
               "greedy_err": rec.get("greedy_err") or ("" if rec["greedy_pass"] else "wrong_answer"),
               "gate": int(rec.get("greedy_gate", False)),
               "case_frac": (rec.get("greedy_npass", 0) / max(rec.get("greedy_ntot", 1), 1)),
               "c": c, "n": n,
               "cost_g": (ptok + gtok) * price / 1000.0,
               "time_g": gen.get("time_greedy", np.nan),
               "time_s1": (gen.get("time_samples", np.nan) / n) if n else np.nan}
        for kk in PASS_KS:
            row[f"p{kk}u"] = upass(c, n, kk)
            row[f"cost_{kk}"] = (ptok + stok_mean * kk) * price / 1000.0
            row[f"time_{kk}"] = row["time_s1"] * kk
        rows.append(row)
tidy = pd.DataFrame(rows)
tidy.to_csv(os.path.join(RESULTS_DIR, "tidy_per_problem.csv"), index=False)

def _lut(col): return {m: dict(zip(g["key"], g[col])) for m, g in tidy.groupby("model")}
PKU = {kk: _lut(f"p{kk}u") for kk in PASS_KS}
G    = _lut("greedy");  GATE = _lut("gate")
CG   = _lut("cost_g");  CK = {kk: _lut(f"cost_{kk}") for kk in PASS_KS}
TG   = _lut("time_g");  TK = {kk: _lut(f"time_{kk}") for kk in PASS_KS}

def boot_ci(vals, B=N_BOOT, seed=123, agg=np.nanmean):
    v = np.asarray(vals, dtype=float); n = len(v)
    rng = np.random.default_rng(seed)
    st = np.array([agg(v[rng.integers(0, n, n)]) for _ in range(B)])
    return float(np.nanpercentile(st, 2.5)), float(np.nanpercentile(st, 97.5))

tab1 = pd.DataFrame([{
    "Model": m["name"], "Params": m["params"], "HF repo": m["repo"],
    "Precision": GEN[m["name"]][keys[0]].get("precision", "bf16"),
    "Rel. price / 1k tok": m["price"],
    "Role": "base (benchmarked, excluded from routing)" if m["base"] else "routing pool"
} for m in MODELS if m["name"] in RAW])
tab1.to_csv(os.path.join(RESULTS_DIR, "tables", "tab1_models.csv"), index=False)
try: tab1.to_latex(os.path.join(RESULTS_DIR, "tables", "tab1_models.tex"), index=False)
except Exception: pass
print(tab1.to_string(index=False))

In [ ]:
# ── 5.3 · Table 2 (overall), 2b (difficulty), 2c (failure mix) ────────────────
rows = []
for m in sorted(RAW, key=lambda m: -np.mean([G[m][k] for k in keys])):
    g = np.mean([G[m][k] for k in keys])
    glo, ghi = wilson(g, len(keys))
    r = {"Model": m + (" (base)" if IS_BASE[m] else ""),
         "greedy pass@1": f"{g:.1%}", "95% CI": f"[{glo:.1%}, {ghi:.1%}]"}
    for kk in PASS_KS:
        v = np.array([PKU[kk][m][k] for k in keys])
        lo, hi = boot_ci(v, B=2000, seed=100 + kk)
        r[f"pass@{kk}"] = f"{np.nanmean(v):.1%}"
        r[f"CI@{kk}"] = f"[{lo:.1%}, {hi:.1%}]"
    r["mean cases passed"] = f"{tidy[tidy.model==m].case_frac.mean():.1%}"
    r["cost@1"] = round(float(np.mean([CG[m][k] for k in keys])), 3)
    r["s/problem"] = round(float(np.nanmean([TG[m][k] for k in keys])), 2)
    rows.append(r)
tab2 = pd.DataFrame(rows)
tab2.to_csv(os.path.join(RESULTS_DIR, "tables", "tab2_overall.csv"), index=False)
try: tab2.to_latex(os.path.join(RESULTS_DIR, "tables", "tab2_overall.tex"), index=False)
except Exception: pass
print("-- TABLE 2 - overall performance --")
print(tab2.to_string(index=False))
print("\n'mean cases passed' is a partial-credit view: if pass@1 is near zero but this is high, "
      "models are close and the binary metric is hiding your signal.")

# ---- Table 2b: difficulty gradient (the load-bearing sanity check on TACO) ----
DIFFS = ["easy", "medium", "medium_hard", "hard", "very_hard", "unknown"]
present = [d for d in DIFFS if sum(1 for k in keys if diff_of[k] == d) > 0]
rows = []
for m in RAW:
    row = {"Model": m}
    for d in present:
        ks = [k for k in keys if diff_of[k] == d]
        p = np.mean([G[m][k] for k in ks])
        lo, hi = wilson(p, len(ks))
        row[f"{d} (n={len(ks)})"] = f"{p:.1%} [{lo:.0%},{hi:.0%}]"
    rows.append(row)
tab2b = pd.DataFrame(rows)
tab2b.to_csv(os.path.join(RESULTS_DIR, "tables", "tab2b_difficulty.csv"), index=False)
print("\n-- TABLE 2b - greedy pass@1 by difficulty (95% Wilson CI) --")
print(tab2b.to_string(index=False))
easy_ks = [k for k in keys if diff_of[k] == "easy"]
if easy_ks:
    best_easy = max(np.mean([G[m][k] for k in easy_ks]) for m in MAIN)
    print(f"\nbest model on easy problems: {best_easy:.1%}")
    if best_easy < 0.10:
        print("WARNING: even easy problems are near zero. That is a harness or prompting "
              "problem, not benchmark difficulty. Inspect Table 2c and some raw generations "
              "BEFORE writing any results.")

# ---- Table 2c: failure composition ------------------------------------------
rows = []
for m in RAW:
    sub = tidy[(tidy.model == m) & (tidy.greedy == 0)]
    cnt = sub.greedy_err.value_counts().to_dict()
    rows.append({"Model": m, "failures": int(len(sub)), **cnt})
tab2c = pd.DataFrame(rows).fillna(0)
for c in tab2c.columns:
    if c != "Model": tab2c[c] = tab2c[c].astype(int)
tab2c.to_csv(os.path.join(RESULTS_DIR, "tables", "tab2c_errors.csv"), index=False)
print("\n-- TABLE 2c - failure composition (greedy) --")
print(tab2c.to_string(index=False))
print("\nHealthy TACO run: failures dominated by wrong_answer and tle. A large syntax_error or "
      "missing_fn share means extraction or prompting is broken, and the scores mean nothing.")

In [ ]:
# ── 5.4 · Table 3 + Figs 1, 2, 7 ──────────────────────────────────────────────
import matplotlib
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 10, "figure.dpi": 120, "savefig.dpi": 300, "savefig.bbox": "tight"})
FIG = os.path.join(RESULTS_DIR, "figures")

mat = (tidy[tidy.key.isin(keys)].pivot_table(index="model", columns="category", values="greedy")
       .reindex(columns=order))
mat = mat.reindex([m for m in RAW if m in mat.index])
(mat * 100).round(2).to_csv(os.path.join(RESULTS_DIR, "tables", "tab3_category_matrix.csv"))
try: (mat * 100).round(1).to_latex(os.path.join(RESULTS_DIR, "tables", "tab3_category_matrix.tex"))
except Exception: pass
print("-- TABLE 3 - greedy pass@1 by category (%) --")
print((mat * 100).round(1).to_string())

data = mat.values * 100
vmax = max(float(np.nanmax(data)), 1.0)          # TACO scores are low: scale to the data
fig, ax = plt.subplots(figsize=(max(9, len(order) * 1.25), 0.45 * len(mat) + 2.2))
im = ax.imshow(data, cmap="YlOrRd", vmin=0, vmax=vmax, aspect="auto")
ax.set_xticks(range(len(order)))
ax.set_xticklabels([f"{sh(c)}\n(n={cat_n[c]})" for c in order], rotation=30, ha="right", fontsize=8)
ax.set_yticks(range(len(mat))); ax.set_yticklabels(mat.index, fontsize=8)
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        if not np.isnan(data[i, j]):
            ax.text(j, i, f"{data[i,j]:.1f}", ha="center", va="center", fontsize=7,
                    color="black" if data[i, j] < 0.65 * vmax else "white")
for j in range(data.shape[1]):
    col = data[:, j]
    if not np.all(np.isnan(col)):
        ax.add_patch(plt.Rectangle((j - .5, int(np.nanargmax(col)) - .5), 1, 1,
                                   fill=False, ec="#1E40AF", lw=2.2))
ax.set_title("TACO greedy pass@1 by category (%) — box = best model per category")
fig.colorbar(im, label="pass@1 (%)")
plt.tight_layout(); plt.savefig(f"{FIG}/fig1_heatmap.png"); plt.show()

fig, ax = plt.subplots(figsize=(max(12, len(order) * 2.0), 5))
nm = len(mat.index); w = 0.82 / max(nm, 1)
for i, m in enumerate(mat.index):
    xs, ys, lo, hi = [], [], [], []
    for j, c in enumerate(order):
        ks = [k for k in keys if cat_of[k] == c]
        p = np.mean([G[m][k] for k in ks]) if ks else np.nan
        l, h = wilson(p, len(ks))
        xs.append(j + (i - nm / 2 + 0.5) * w); ys.append(p * 100)
        lo.append(max((p - l) * 100, 0)); hi.append(max((h - p) * 100, 0))
    ax.bar(xs, ys, width=w * 0.9, label=m, alpha=0.9)
    ax.errorbar(xs, ys, yerr=[lo, hi], fmt="none", ecolor="#333", elinewidth=0.6, capsize=1.2)
ax.set_xticks(range(len(order)))
ax.set_xticklabels([f"{sh(c)}\n(n={cat_n[c]})" for c in order], fontsize=8)
ax.set_ylabel("greedy pass@1 (%)")
ax.set_title("Per-category pass@1 (95% Wilson CI) — note the CI width at these sample sizes")
ax.legend(fontsize=7, ncol=3); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.savefig(f"{FIG}/fig2_categories.png"); plt.show()

# Fig 7 - failure composition
etypes = [c for c in tab2c.columns if c not in ("Model", "failures")]
ecolors = {"syntax_error": "#DC2626", "wrong_answer": "#F59E0B", "runtime_error": "#8B5CF6",
           "tle": "#0EA5E9", "missing_fn": "#065F46"}
fig, ax = plt.subplots(figsize=(max(8, len(tab2c) * 0.9), 4.6))
bottom = np.zeros(len(tab2c))
for e in etypes:
    v = tab2c[e].values.astype(float)
    ax.bar(tab2c.Model, v, bottom=bottom, label=e, color=ecolors.get(e, "#94A3B8"), alpha=0.9)
    bottom += v
ax.set_ylabel("# failed problems (greedy)"); ax.legend(fontsize=8)
ax.set_title("Failure composition by model")
plt.xticks(rotation=25, ha="right", fontsize=8); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.savefig(f"{FIG}/fig7_errors.png"); plt.show()
print("figs 1, 2, 7 saved")

## 6 · Pairwise significance (Table 4)

Exact McNemar on paired greedy outcomes within each category, corrected across the whole family
of tests with both Benjamini-Hochberg (FDR) and Holm (FWER). At TACO's solve rates most pairs
will have very few discordant problems, so expect few survivors — that is information, not a
failure, and the paper must report it as such.

In [ ]:
# ── 6.1 · Table 4: McNemar + BH/Holm + Cohen's h ──────────────────────────────
from scipy.stats import binomtest
from statsmodels.stats.multitest import multipletests
from itertools import combinations

def mcnemar_p(a, b):
    a, b = np.asarray(a).astype(int), np.asarray(b).astype(int)
    b10 = int(((a == 1) & (b == 0)).sum()); b01 = int(((a == 0) & (b == 1)).sum())
    nd = b10 + b01
    if nd == 0: return 1.0, b10, b01
    return binomtest(min(b10, b01), nd, 0.5).pvalue, b10, b01

def cohens_h(p1, p2):
    f = lambda p: 2 * math.asin(math.sqrt(min(max(p, 0.0), 1.0)))
    return f(p1) - f(p2)

rows, skipped = [], []
for c in order:
    ks = [k for k in keys if cat_of[k] == c]
    if len(ks) < MIN_CAT_N_FOR_TESTS:
        skipped.append((c, len(ks))); continue
    for ma, mb in combinations(MAIN, 2):
        a = np.array([G[ma][k] for k in ks]); b = np.array([G[mb][k] for k in ks])
        p, w1, w2 = mcnemar_p(a, b)
        rows.append({"Category": c, "n": len(ks), "Model A": ma, "Model B": mb,
                     "A only": w1, "B only": w2, "discordant": w1 + w2,
                     "pass@1 A": round(a.mean(), 3), "pass@1 B": round(b.mean(), 3),
                     "p_raw": p, "Cohen h": round(abs(cohens_h(a.mean(), b.mean())), 3)})
if skipped:
    print("descriptive only (n below the testing threshold): " +
          ", ".join(f"{c} (n={n})" for c, n in skipped))
if rows:
    st = pd.DataFrame(rows)
    st["q_BH"] = multipletests(st.p_raw, method="fdr_bh")[1]
    st["p_Holm"] = multipletests(st.p_raw, method="holm")[1]
    st["sig_BH"] = np.where(st.q_BH < ALPHA, "yes", "-")
    st["sig_Holm"] = np.where(st.p_Holm < ALPHA, "yes", "-")
    st.sort_values("q_BH").to_csv(os.path.join(RESULTS_DIR, "tables", "tab4_pairwise_stats.csv"),
                                  index=False)
    try: st.round(4).to_latex(os.path.join(RESULTS_DIR, "tables", "tab4_pairwise_stats.tex"),
                              index=False)
    except Exception: pass
    print(f"\n-- TABLE 4 - {int((st.q_BH<ALPHA).sum())}/{len(st)} pairs significant after BH, "
          f"{int((st.p_Holm<ALPHA).sum())} after Holm --")
    print(f"median discordant pairs per test: {int(st.discordant.median())}")
    print(st.sort_values("q_BH").head(15).to_string(index=False))
    print("\nREAD THIS: with few discordant problems these tests have almost no power. If nothing "
          "survives correction, the honest claim is that per-category differences are not "
          "established at this sample size -- not that they are absent.")
else:
    print("No category large enough to test. Report Table 3 descriptively.")

## 7 · Cross-validated routing and equal-budget policies

The route map and the best-single baseline are fit on 9 folds and applied to the held-out fold,
so no problem contributes to its own routing decision.

**Cascades are gated honestly.** A cascade escalates while the first `PUBLIC_GATE_K` cases fail
and deploys the first model that passes them; success is then graded on *all* cases. Gating on
the full grading set — what the original notebooks did implicitly — inflates cascade performance
by letting the policy peek at its own metric.

In [ ]:
# ── 7.1 · Cross-validated route map ───────────────────────────────────────────
kidx = {k: i for i, k in enumerate(keys)}
rng0 = np.random.default_rng(GLOBAL_SEED + 1)
fold_of = {}
for c in cats:                                    # stratified by category
    ks = sorted([k for k in keys if cat_of[k] == c]); rng0.shuffle(ks)
    for i, k in enumerate(ks): fold_of[k] = i % CV_FOLDS
folds = [[k for k in keys if fold_of[k] == f] for f in range(CV_FOLDS)]

def fit_route_map(train, catmap=None):
    """Best model per category on TRAIN only (unbiased pass@1; ties -> cheaper).
    catmap lets §7.6 re-fit under a permuted taxonomy without touching anything else."""
    cm = catmap or cat_of
    glob_best = max(MAIN, key=lambda m: (np.mean([PKU[1][m][k] for k in train]), -PRICE[m]))
    mp = {}
    for c in sorted(set(cm.values())):
        tks = [k for k in train if cm[k] == c]
        if not tks: continue
        rank = sorted(MAIN, key=lambda m: (-np.mean([PKU[1][m][k] for k in tks]), PRICE[m]))
        s = lambda m: np.mean([PKU[1][m][k] for k in tks])
        margin = s(rank[0]) - (s(rank[1]) if len(rank) > 1 else s(rank[0]))
        mp[c] = rank[0] if margin >= ROUTE_MIN_MARGIN else glob_best
    return mp, glob_best

route_assign, bestsingle_assign = {}, {}
stability = defaultdict(Counter); bs_counter = Counter()
for f in range(CV_FOLDS):
    test = folds[f]; train = [k for k in keys if fold_of[k] != f]
    mp, glob_best = fit_route_map(train)
    for c, m in mp.items(): stability[c][m] += 1
    bs = max(MAIN, key=lambda m: (np.mean([PKU[1][m][k] for k in train]), -PRICE[m]))
    bs_counter[bs] += 1
    for k in test:
        route_assign[k] = mp.get(cat_of[k], glob_best)
        bestsingle_assign[k] = bs
BS_NAME = bs_counter.most_common(1)[0][0]

stab = pd.DataFrame([{
    "Category": c, "n": cat_n[c], "Modal routed model": stability[c].most_common(1)[0][0],
    "Fold agreement": f"{stability[c].most_common(1)[0][1]/CV_FOLDS:.0%}",
    "Distinct models across folds": len(stability[c])} for c in order if c in stability])
stab.to_csv(os.path.join(RESULTS_DIR, "tables", "tab9_route_stability.csv"), index=False)
print(f"best-single baseline selected in {bs_counter[BS_NAME]}/{CV_FOLDS} folds: {BS_NAME}\n")
print("-- route-map stability across folds --"); print(stab.to_string(index=False))
print("\nREAD THIS: low fold agreement means the category winner is resampling noise. This table "
      "is the honest answer to 'does a category route map exist at all?' -- report it.")

In [ ]:
# ── 7.2 · Policies (cascades gated on public cases, graded on all) ────────────
def vec(fn): return np.array([fn(k) for k in keys], dtype=float)
ORDER_CHEAP = sorted(MAIN, key=lambda m: PRICE[m])
policies, POLICY_GENS = {}, {}

for m in MAIN:
    policies[f"single:{m}@1"] = (vec(lambda k, m=m: G[m][k]), vec(lambda k, m=m: CG[m][k]),
                                 vec(lambda k, m=m: TG[m][k]))
    POLICY_GENS[f"single:{m}@1"] = 1.0
policies["best-single(CV)@1"] = (vec(lambda k: G[bestsingle_assign[k]][k]),
                                 vec(lambda k: CG[bestsingle_assign[k]][k]),
                                 vec(lambda k: TG[bestsingle_assign[k]][k]))
policies["CV-route@1"] = (vec(lambda k: G[route_assign[k]][k]),
                          vec(lambda k: CG[route_assign[k]][k]),
                          vec(lambda k: TG[route_assign[k]][k]))
RESAMPLE_NAME = f"resample{RESAMPLE_K}:best-single"
ROUTE_K_NAME  = f"CV-route@{RESAMPLE_K}"
policies[RESAMPLE_NAME] = (vec(lambda k: PKU[RESAMPLE_K][bestsingle_assign[k]][k]),
                           vec(lambda k: CK[RESAMPLE_K][bestsingle_assign[k]][k]),
                           vec(lambda k: TK[RESAMPLE_K][bestsingle_assign[k]][k]))
policies[ROUTE_K_NAME] = (vec(lambda k: PKU[RESAMPLE_K][route_assign[k]][k]),
                          vec(lambda k: CK[RESAMPLE_K][route_assign[k]][k]),
                          vec(lambda k: TK[RESAMPLE_K][route_assign[k]][k]))
POLICY_GENS.update({"best-single(CV)@1": 1.0, "CV-route@1": 1.0,
                    RESAMPLE_NAME: float(RESAMPLE_K), ROUTE_K_NAME: float(RESAMPLE_K)})

def simulate_cascade(seq_for_key, noise=None):
    """Escalate in cost order while the PUBLIC gate fails; grade the deployed model on ALL
    cases. noise = {model: price multiplier} for the sensitivity analysis."""
    mult = (lambda m: noise.get(m, 1.0)) if noise else (lambda m: 1.0)
    s, c, t, g = (np.zeros(len(keys)) for _ in range(4))
    for k in keys:
        seq = seq_for_key(k)
        cost = tim = 0.0; used = 0; deployed = None
        for m in seq:
            cost += CG[m][k] * mult(m); tim += TG[m][k]; used += 1; deployed = m
            if GATE[m][k]:                      # public cases say "ship it"
                break
        i = kidx[k]
        s[i] = G[deployed][k] if deployed else 0    # graded on all cases
        c[i], t[i], g[i] = cost, tim, used
    return s, c, t, g

DEPTH = min(5, len(ORDER_CHEAP))
_s, _c, _t, _g = simulate_cascade(lambda k: ORDER_CHEAP[:DEPTH])
policies[f"cascade@{DEPTH}"] = (_s, _c, _t); POLICY_GENS[f"cascade@{DEPTH}"] = _g

def smart_cascade_cv():
    thr_sweep = [0.0, 0.01, 0.02, 0.05, 0.08, 0.12, 0.20]
    s, c, t, g = (np.zeros(len(keys)) for _ in range(4))
    chosen = []
    for f in range(CV_FOLDS):
        test, train = folds[f], [k for k in keys if fold_of[k] != f]
        prior = {(m, cc): float(np.mean([PKU[1][m][k] for k in train if cat_of[k] == cc] or [0.0]))
                 for m in MAIN for cc in cats}
        def seq(thr):
            return lambda k: ([m for m in ORDER_CHEAP if prior[(m, cat_of[k])] >= thr]
                              or ORDER_CHEAP[:1])
        def score(thr, ks):
            sub = [kidx[k] for k in ks]
            ss, cc, _, _ = simulate_cascade(seq(thr))
            return np.mean(ss[sub]), -np.mean(cc[sub])
        best = max(thr_sweep, key=lambda thr: score(thr, train))   # tuned on TRAIN only
        chosen.append(best)
        ss, cc, tt, gg = simulate_cascade(seq(best))
        for k in test:
            i = kidx[k]; s[i], c[i], t[i], g[i] = ss[i], cc[i], tt[i], gg[i]
    print(f"  smart-cascade threshold per fold: {chosen}")
    return s, c, t, g
_s, _c, _t, _g = smart_cascade_cv()
policies["smart-cascade(CV)"] = (_s, _c, _t); POLICY_GENS["smart-cascade(CV)"] = _g

policies["oracle@1"] = (vec(lambda k: int(any(G[m][k] for m in MAIN))),
                        vec(lambda k: min([CG[m][k] for m in MAIN if G[m][k]]
                                          or [min(CG[m][k] for m in MAIN)])),
                        vec(lambda k: min([TG[m][k] for m in MAIN if G[m][k]]
                                          or [min(TG[m][k] for m in MAIN)])))
policies["ceiling(any model, any sample)"] = (
    vec(lambda k: int(any(any(RAW[m][k]["sample_pass"]) for m in MAIN))),
    np.full(len(keys), np.nan), np.full(len(keys), np.nan))
POLICY_GENS.update({"oracle@1": float(len(MAIN)),
                    "ceiling(any model, any sample)": float(N_SAMPLES * len(MAIN))})

gate_rate = np.mean([GATE[bestsingle_assign[k]][k] for k in keys])
solve_given_gate = np.mean([G[bestsingle_assign[k]][k] for k in keys
                            if GATE[bestsingle_assign[k]][k]] or [np.nan])
print(f"\ndefined {len(policies)} policies over {len(keys)} problems")
print(f"public-gate pass rate (best-single): {gate_rate:.1%}; of those, {solve_given_gate:.1%} "
      f"also pass all cases -> report both, it quantifies how much the gate over-promises")

In [ ]:
# ── 7.3 · Table 5: policies with bootstrap CIs, paired vs best-single ─────────
def paired_delta(sA, sB, B=N_BOOT, seed=11):
    d = np.asarray(sA) - np.asarray(sB)
    rng = np.random.default_rng(seed); n = len(d)
    bs = np.array([d[rng.integers(0, n, n)].mean() for _ in range(B)])
    return float(d.mean()), (float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5)))

def boot2(s, c, B=2000, seed=7):
    rng = np.random.default_rng(seed); n = len(s); out = np.empty((B, 2))
    s, c = np.asarray(s), np.asarray(c)
    for b in range(B):
        i = rng.integers(0, n, n)
        out[b] = (np.nanmean(s[i]), np.nanmean(c[i]))
    return out

real5 = {RESAMPLE_NAME:
             np.array([int(any(RAW[bestsingle_assign[k]][k]["sample_pass"][:RESAMPLE_K])) for k in keys]),
         ROUTE_K_NAME:
             np.array([int(any(RAW[route_assign[k]][k]["sample_pass"][:RESAMPLE_K])) for k in keys])}
ref = policies["best-single(CV)@1"][0]

rows = []
for name, (s, c, t) in policies.items():
    b = boot2(s, c)
    d, ci = paired_delta(s, ref)
    binA = real5.get(name, (np.asarray(s) > 0.5).astype(int)
                     if set(np.unique(s)) <= {0.0, 1.0} else None)
    pmc = mcnemar_p(binA, ref.astype(int))[0] if binA is not None else float("nan")
    rows.append({"Policy": name,
                 "Gens/problem": round(float(np.mean(POLICY_GENS.get(name, 1.0))), 2),
                 "Solve rate": f"{np.nanmean(s):.1%}",
                 "95% CI": f"[{np.nanpercentile(b[:,0],2.5):.1%}, {np.nanpercentile(b[:,0],97.5):.1%}]",
                 "Cost": round(float(np.nanmean(c)), 3),
                 "Cost CI": f"[{np.nanpercentile(b[:,1],2.5):.2f}, {np.nanpercentile(b[:,1],97.5):.2f}]",
                 "s/problem": round(float(np.nanmean(t)), 2),
                 "Solve/Cost": round(float(np.nanmean(s) / max(np.nanmean(c), 1e-9)), 2),
                 "delta vs best-single": f"{d:+.1%}",
                 "delta 95% CI": f"[{ci[0]:+.1%}, {ci[1]:+.1%}]",
                 "McNemar p": (f"{pmc:.4f}" if pmc == pmc else "-"),
                 "Sig": "yes" if (ci[0] > 0 or ci[1] < 0) else "-"})
tab5 = pd.DataFrame(rows).sort_values("Cost")
tab5.to_csv(os.path.join(RESULTS_DIR, "tables", "tab5_policies.csv"), index=False)
try: tab5.to_latex(os.path.join(RESULTS_DIR, "tables", "tab5_policies.tex"), index=False)
except Exception: pass
print("-- TABLE 5 - equal-budget policies (paired bootstrap vs CV best-single) --")
print(tab5.to_string(index=False))
print("\nREAD THIS: 'CV-route@1' vs 'best-single(CV)@1' is the routing claim. If the delta CI "
      "spans 0, write that routing does not significantly beat the best single model, and use "
      "the oracle row to quantify the headroom a better router would have.")

In [ ]:
# ── 7.4 · Figs 3-6 ────────────────────────────────────────────────────────────
def pcolor(n):
    for pat, col in [("best-single", "#0F172A"), ("resample", "#7C3AED"), ("CV-route", "#D97706"),
                     ("smart", "#16A34A"), ("cascade", "#DC2626"), ("oracle", "#2563EB"),
                     ("ceiling", "#64748B"), ("single:", "#94A3B8")]:
        if pat in n: return col
    return "#888888"

fig, ax = plt.subplots(figsize=(9.5, 6))
for name, (s, c, t) in policies.items():
    if np.all(np.isnan(c)): continue
    b = boot2(s, c, B=1500, seed=3)
    x, y = np.nanmean(c), np.nanmean(s) * 100
    ax.errorbar(x, y,
                xerr=[[x - np.nanpercentile(b[:, 1], 2.5)], [np.nanpercentile(b[:, 1], 97.5) - x]],
                yerr=[[y - np.nanpercentile(b[:, 0], 2.5) * 100],
                      [np.nanpercentile(b[:, 0], 97.5) * 100 - y]],
                fmt="o", ms=6, color=pcolor(name), capsize=2.5, alpha=0.9,
                label=None if name.startswith("single:") else name)
    if not name.startswith("single:"):
        ax.annotate(name, (x, y), fontsize=6.5, xytext=(4, 4), textcoords="offset points")
ax.set_xlabel("mean relative token cost per problem"); ax.set_ylabel("solve rate (%)")
ax.set_title("Accuracy–cost Pareto, TACO (95% bootstrap CIs); grey = individual models @1")
ax.grid(alpha=0.3); ax.legend(fontsize=7, loc="upper left")
plt.tight_layout(); plt.savefig(f"{FIG}/fig3_pareto.png"); plt.show()

sel = [n for n in [RESAMPLE_NAME, ROUTE_K_NAME, f"cascade@{DEPTH}", "smart-cascade(CV)"]
       if n in policies]
fig, ax = plt.subplots(figsize=(8.5, 5))
for i, name in enumerate(sel):
    s, c, _ = policies[name]
    b = boot2(s, c, B=3000, seed=5)[:, 0] * 100
    v = np.nanmean(s) * 100
    ax.bar(i, v, 0.6, color=pcolor(name), alpha=0.9)
    ax.errorbar(i, v, yerr=[[max(v - np.nanpercentile(b, 2.5), 0)],
                            [max(np.nanpercentile(b, 97.5) - v, 0)]],
                fmt="none", ecolor="#111", capsize=5)
    ax.text(i, v, f"  {v:.1f}%", ha="center", va="bottom", fontsize=9)
ax.set_xticks(range(len(sel)))
ax.set_xticklabels([s.replace(":", "\n").replace("@", "\n@") for s in sel], fontsize=8)
ax.set_ylabel("solve rate (%)"); ax.grid(axis="y", alpha=0.3)
ax.set_title("Width (resample one model) vs depth (cascade across models), matched budget")
plt.tight_layout(); plt.savefig(f"{FIG}/fig4_width_vs_depth.png"); plt.show()

sel = ["best-single(CV)@1", "CV-route@1", "oracle@1"]
fig, ax = plt.subplots(figsize=(7.5, 5))
for i, name in enumerate(sel):
    s = policies[name][0]
    b = boot2(s, np.zeros_like(s), B=3000, seed=9)[:, 0] * 100
    v = np.nanmean(s) * 100
    ax.bar(i, v, 0.55, color=pcolor(name), alpha=0.9)
    ax.errorbar(i, v, yerr=[[max(v - np.nanpercentile(b, 2.5), 0)],
                            [max(np.nanpercentile(b, 97.5) - v, 0)]],
                fmt="none", ecolor="#111", capsize=5)
    ax.text(i, v, f"  {v:.1f}%", ha="center", va="bottom", fontsize=10)
ax.set_xticks(range(3))
ax.set_xticklabels(["best single model\n(CV-selected)", "category routing\n(CV)",
                    "oracle\n(per-problem)"])
ax.set_ylabel("greedy pass@1 (%)"); ax.grid(axis="y", alpha=0.3)
ax.set_title(f"Does category routing beat the best single model? ({CV_FOLDS}-fold CV, 95% CI)")
plt.tight_layout(); plt.savefig(f"{FIG}/fig5_routing.png"); plt.show()

solved_by = np.array([sum(G[m][k] for m in MAIN) for k in keys])
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
axes[0].hist(solved_by, bins=np.arange(-0.5, len(MAIN) + 1.5), color="#2563EB", alpha=0.85)
axes[0].set_xlabel("# models solving the problem (greedy)"); axes[0].set_ylabel("# problems")
axes[0].set_title(f"Complementarity: {int((solved_by==0).sum())} solved by none, "
                  f"{int((solved_by==len(MAIN)).sum())} by all")
axes[0].set_yscale("log")
uw = Counter()
for k in keys:
    sv = [m for m in MAIN if G[m][k]]
    if len(sv) == 1: uw[sv[0]] += 1
items = sorted(uw.items(), key=lambda kv: -kv[1])
axes[1].bar([x[0] for x in items], [x[1] for x in items], color="#16A34A", alpha=0.85)
axes[1].set_ylabel("# problems solved ONLY by this model")
axes[1].set_title("Unique wins per model")
axes[1].tick_params(axis="x", rotation=40, labelsize=7)
for a in axes: a.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.savefig(f"{FIG}/fig6_complementarity.png"); plt.show()
print("figs 3-6 saved")

In [ ]:
# ── 7.5 · Table 6: cost-model robustness + wall-clock axis ────────────────────
core = [n for n in ["best-single(CV)@1", "CV-route@1", RESAMPLE_NAME,
                    f"cascade@{DEPTH}", "smart-cascade(CV)"] if n in policies]
DRAWS = 200
wins = Counter()
for t in range(DRAWS):
    rng = np.random.default_rng(9000 + t)
    noise = {m: float(rng.lognormal(0, 0.4)) for m in MAIN}    # ~+-50% price mis-specification
    def perturbed(name):
        if name == "best-single(CV)@1":
            return np.mean([CG[bestsingle_assign[k]][k] * noise[bestsingle_assign[k]] for k in keys])
        if name == "CV-route@1":
            return np.mean([CG[route_assign[k]][k] * noise[route_assign[k]] for k in keys])
        if name == RESAMPLE_NAME:
            return np.nanmean([CK[RESAMPLE_K][bestsingle_assign[k]][k] * noise[bestsingle_assign[k]]
                               for k in keys])
        return float(np.mean(simulate_cascade(lambda k: ORDER_CHEAP[:DEPTH], noise=noise)[1]))
    wins[max(core, key=lambda n: np.nanmean(policies[n][0]) / max(perturbed(n), 1e-9))] += 1
tab6 = pd.DataFrame([{"Policy": n, "wins": w, "share": round(w / DRAWS, 3)}
                     for n, w in wins.most_common()])
tab6.to_csv(os.path.join(RESULTS_DIR, "tables", "tab6_cost_sensitivity.csv"), index=False)
print(f"-- TABLE 6 - solve-per-cost winner under {DRAWS} price perturbations (lognormal sigma=0.4) --")
print(tab6.to_string(index=False))
print("\nwall-clock axis (mean s/problem, price-independent):")
for n in core: print(f"  {n:<26} {np.nanmean(policies[n][2]):.2f}s")

In [ ]:
# ── 7.6 · Table 8: taxonomy priority-permutation sensitivity ──────────────────
TAGS = {k: PROBS[k]["tags"] for k in keys}
base_map, _ = fit_route_map(keys)
base_solve = float(np.mean([G[base_map[cat_of[k]]][k] for k in keys if cat_of[k] in base_map]))
rngp = np.random.default_rng(GLOBAL_SEED + 77)

rows = []
for t in range(TAXONOMY_PERMS):
    prio = list(CATEGORY_PRIORITY); rngp.shuffle(prio)
    perm_cat = {}
    for k in keys:
        nc = assign_category(matched_categories(TAGS[k]), prio)
        perm_cat[k] = nc or cat_of[k]
    reassigned = sum(1 for k in keys if perm_cat[k] != cat_of[k])
    mp, _ = fit_route_map(keys, catmap=perm_cat)          # non-CV: compares maps, not performance
    agree = np.mean([mp.get(perm_cat[k]) == base_map.get(cat_of[k]) for k in keys])
    solve = np.mean([G[mp[perm_cat[k]]][k] for k in keys if perm_cat[k] in mp])
    rows.append({"perm": t, "% problems reassigned": round(100 * reassigned / len(keys), 1),
                 "routed-model agreement": round(float(agree), 3),
                 "non-CV route solve": round(float(solve), 4)})
tab8 = pd.DataFrame(rows)
summary = {"baseline_non_cv_solve": round(base_solve, 4),
           "mean_solve": round(float(tab8["non-CV route solve"].mean()), 4),
           "sd_solve": round(float(tab8["non-CV route solve"].std()), 4),
           "min_solve": round(float(tab8["non-CV route solve"].min()), 4),
           "max_solve": round(float(tab8["non-CV route solve"].max()), 4),
           "mean_agreement": round(float(tab8["routed-model agreement"].mean()), 3)}
tab8.to_csv(os.path.join(RESULTS_DIR, "tables", "tab8_taxonomy_sensitivity.csv"), index=False)
json.dump(summary, open(os.path.join(RESULTS_DIR, "taxonomy_sensitivity_summary.json"), "w"), indent=2)
print(f"-- TABLE 8 - taxonomy priority-permutation sensitivity ({TAXONOMY_PERMS} permutations) --")
print(json.dumps(summary, indent=2))
print("\nThese are non-CV numbers by design: they measure how much the route MAP moves when the "
      "tie-breaking order changes, not held-out performance. If solve swings widely across "
      "permutations, your category axis is partly an artifact of the priority list -- say so.")

## 8 · Repair vs blind resampling at equal budget (Table 7)

This replaces the Phase-2 agent. The publishable question is not "does our agent work" but
**"at the same number of generations, does error-feedback repair beat sampling again?"**

TACO makes this arm stronger than it was on HumanEval: the executor returns the actual failing
input, the expected output and what the model produced, so the feedback is concrete rather than
"your code failed".

- **Repair**: attempt 1 is the stored greedy generation (free). Attempts 2..`REPAIR_ATTEMPTS`
  re-prompt the *same routed model* with the first failure. Rounds are batched across problems.
- **Baseline**: unbiased pass@3 of the same routed model — blind resampling at the same budget.
- Early stop when two consecutive attempts produce the same error signature (the model is
  looping), which is itself worth reporting.

In [ ]:
# ── 8.1 · Repair loop (round-based, resumable) ────────────────────────────────
def repair_prompt(prob, code_str, detail):
    fb = "Your solution failed the tests."
    if detail.get("args"):     fb += f"\nFailing input:\n{detail['args']}"
    if detail.get("expected"): fb += f"\nExpected output: {detail['expected']}"
    if detail.get("got"):      fb += f"\nYour output: {detail['got']}"
    if detail.get("err"):      fb += f"\nError: {detail['err']}"
    if detail.get("type") == "tle":
        fb += "\nYour solution was too slow. Use a more efficient algorithm."
    return (f"{task_text(prob)}\n\nYour previous attempt:\n```python\n{code_str}\n```\n\n{fb}\n\n"
            "Return a corrected, complete solution. Output ONLY raw Python code.")

def run_repair_for_model(mname, task_keys):
    path = jsonl_path("repair", mname)
    state = jsonl_load(path)
    todo = [k for k in task_keys if k not in state]
    if not todo:
        print(f"  {mname:<22} repair complete ({len(state)})"); return
    cfg = next(c for c in MODELS if c["name"] == mname)
    live = {}
    for k in todo:                                   # attempt 1 = the stored greedy generation
        code0 = GEN[mname][k]["greedy_code"]
        ok, cases, et, det = run_code(PROBS[k], code0)
        live[k] = {"key": k, "model": mname, "category": cat_of[k], "difficulty": diff_of[k],
                   "solved": bool(ok), "gens_used": 1,
                   "attempts": [{"pass": bool(ok), "err": et}],
                   "code": code0, "detail": det, "stopped": "solved" if ok else ""}
    pend = [k for k in todo if not live[k]["solved"]]
    print(f"  {mname:<22} {len(todo)-len(pend)}/{len(todo)} solved by greedy; repairing {len(pend)}")
    eng = (make_engine(cfg) if pend else None)
    try:
        for rnd in range(2, REPAIR_ATTEMPTS + 1):
            if not pend: break
            items = []
            for k in pend:
                p = dict(PROBS[k])
                p["question"] = repair_prompt(PROBS[k], live[k]["code"], live[k]["detail"])
                p["starter"] = ""          # the repair prompt already carries the context
                items.append((k, p))
            outs = eng.generate(items, 1, SAMPLE_TEMP)
            nxt = []
            for k, o in zip(pend, outs):
                new_code = o["codes"][0] if o["codes"] else live[k]["code"]
                ok, cases, et, det = run_code(PROBS[k], new_code)
                prev_sig = (live[k]["attempts"][-1]["err"],
                            str(live[k]["detail"].get("err", ""))[:60])
                new_sig = (et, str(det.get("err", ""))[:60])
                live[k]["attempts"].append({"pass": bool(ok), "err": et})
                live[k]["gens_used"] += 1
                live[k]["code"], live[k]["detail"], live[k]["solved"] = new_code, det, bool(ok)
                if ok:                    live[k]["stopped"] = "solved"
                elif new_sig == prev_sig: live[k]["stopped"] = "looping"
                else:                     nxt.append(k)
            pend = nxt
            print(f"    round {rnd}: {sum(1 for k in todo if live[k]['solved'])}/{len(todo)} solved")
    finally:
        if eng is not None: eng.close()
    for k in todo:
        rec = live[k]
        rec["final_code"] = rec.pop("code")[:2000]; rec.pop("detail", None)
        jsonl_append(path, rec)

if RUN_REPAIR_ARM and not NOT_RESULTS:
    bym = defaultdict(list)
    for k in keys: bym[route_assign[k]].append(k)
    print(f"repair arm over {len(keys)} problems, {len(bym)} routed models")
    for mname, ks in bym.items():
        try:
            run_repair_for_model(mname, ks)
        except Exception:
            import traceback; print(f"!! repair failed for {mname}, continuing"); traceback.print_exc()
elif RUN_REPAIR_ARM:
    print("repair arm skipped (TRIAL/QUICK/DRY). Run in full LIVE mode for Table 7.")

In [ ]:
# ── 8.2 · Table 7: repair vs blind resample at the same budget ────────────────
rep = {}
for name in ENABLED:
    rep.update(jsonl_load(jsonl_path("repair", name)))
ks = [k for k in keys if k in rep]
if ks and len(ks) >= 0.9 * len(keys):
    s_rep = np.array([int(rep[k]["solved"]) for k in ks])
    s_res = np.array([PKU[REPAIR_K][route_assign[k]][k] for k in ks])   # unbiased pass@k, same model
    gens  = np.array([rep[k]["gens_used"] for k in ks], dtype=float)
    greedy_only = np.array([int(rep[k]["attempts"][0]["pass"]) for k in ks])
    real3 = np.array([int(any(RAW[route_assign[k]][k]["sample_pass"][:REPAIR_K])) for k in ks])
    d, ci = paired_delta(s_rep, s_res, seed=21)
    p, w1, w2 = mcnemar_p(s_rep, real3)
    loops = sum(1 for k in ks if rep[k].get("stopped") == "looping")
    tab7 = pd.DataFrame([
        {"Policy": "greedy only (1 generation)", "Solve rate": f"{greedy_only.mean():.1%}",
         "Gens/problem": 1.0},
        {"Policy": f"blind resample@{REPAIR_K} (same routed model)",
         "Solve rate": f"{np.nanmean(s_res):.1%}", "Gens/problem": float(REPAIR_K)},
        {"Policy": f"error-feedback repair@{REPAIR_ATTEMPTS}", "Solve rate": f"{s_rep.mean():.1%}",
         "Gens/problem": round(float(gens.mean()), 2)}])
    tab7.to_csv(os.path.join(RESULTS_DIR, "tables", "tab7_repair.csv"), index=False)
    try: tab7.to_latex(os.path.join(RESULTS_DIR, "tables", "tab7_repair.tex"), index=False)
    except Exception: pass
    print("-- TABLE 7 - repair vs blind resample --")
    print(tab7.to_string(index=False))
    print(f"\npaired delta (repair - resample): {d:+.1%}  95% CI [{ci[0]:+.1%}, {ci[1]:+.1%}]"
          f"   McNemar p={p:.4f} (against the any-of-3 realization)")
    print(f"repair averaged {gens.mean():.2f} generations/problem; resample always spends {REPAIR_K}")
    print(f"{loops}/{len(ks)} problems stopped early on a repeated error signature")
    # repair helps most where feedback is informative -- worth a sentence in the paper
    by_err = defaultdict(lambda: [0, 0])
    for k in ks:
        e = rep[k]["attempts"][0]["err"] or "solved_first_try"
        by_err[e][0] += int(rep[k]["solved"]); by_err[e][1] += 1
    print("\nrepair success by the error type of attempt 1:")
    for e, (nsolved, ntot) in sorted(by_err.items(), key=lambda kv: -kv[1][1]):
        print(f"  {e:<18} {nsolved}/{ntot}  ({nsolved/max(ntot,1):.0%})")
    print("\nREAD THIS: if the CI spans 0, error-feedback repair does not beat blind resampling "
          "at equal budget -- a publishable negative result, and consistent with the Self-Refine "
          "follow-up literature on tasks with a strong external verifier.")
else:
    print("Repair arm incomplete -- Table 7 skipped.")

## 9 · Manifest, headline numbers, and what you may claim

In [ ]:
# ── 9.1 · Headline numbers + reproducibility manifest ─────────────────────────
import hashlib, platform, datetime, subprocess

def _sha(p, n=16):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""): h.update(chunk)
    return h.hexdigest()[:n]

def _mean(name): return float(np.nanmean(policies[name][0]))
headline = {
    "benchmark": "TACO", "n_problems": len(keys), "n_models_benchmarked": len(RAW),
    "n_models_in_routing_pool": len(MAIN), "base_models_excluded": BASE,
    "date": datetime.datetime.now().isoformat(timespec="seconds"),
    "partial_coverage": PARTIAL, "not_results": NOT_RESULTS,
    "protocol": {"greedy_temp": GREEDY_TEMP, "sample_temp": SAMPLE_TEMP, "top_p": TOP_P,
                 "n_samples": N_SAMPLES, "pass_ks": PASS_KS, "max_new_tokens": MAX_NEW_TOKENS,
                 "precision_mode": PRECISION_MODE, "backend": BACKEND,
                 "grading": f"all cases up to {MAX_TEST_CASES}",
                 "cascade_gate": f"first {PUBLIC_GATE_K} cases; graded on all",
                 "per_case_timeout_s": PER_CASE_TIMEOUT, "cv_folds": CV_FOLDS, "n_boot": N_BOOT},
    "best_single_cv": BS_NAME,
    "best_single_pass1": _mean("best-single(CV)@1"),
    "cv_route_pass1": _mean("CV-route@1"),
    "routing_delta_pp": (_mean("CV-route@1") - _mean("best-single(CV)@1")) * 100,
    "oracle_pass1": _mean("oracle@1"),
    "oracle_gap_pp": (_mean("oracle@1") - _mean("best-single(CV)@1")) * 100,
    f"resample{RESAMPLE_K}_best_single": _mean(RESAMPLE_NAME),
    f"cv_route_at{RESAMPLE_K}": _mean(ROUTE_K_NAME),
    f"cascade@{DEPTH}": _mean(f"cascade@{DEPTH}"),
    "smart_cascade_cv": _mean("smart-cascade(CV)"),
    "ceiling": _mean("ceiling(any model, any sample)"),
    "public_gate_pass_rate": float(gate_rate),
    "taxonomy_sensitivity": summary,
}
json.dump(headline, open(os.path.join(RESULTS_DIR, "headline_numbers.json"), "w"), indent=2)

files = {}
for f in ["corpus.json", "tidy_per_problem.csv", "taxonomy_audit.json"]:
    p = os.path.join(RESULTS_DIR, f)
    if os.path.exists(p): files[f] = _sha(p)
for m in RAW:
    for stage in ("gen", "grade"):
        p = jsonl_path(stage, m)
        if os.path.exists(p): files[f"{stage}/{os.path.basename(p)}"] = _sha(p)
try:
    freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"],
                            capture_output=True, text=True, timeout=120).stdout
except Exception:
    freeze = ""
manifest = {
    "python": platform.python_version(), "platform": platform.platform(),
    "torch": getattr(torch, "__version__", None) if HAS_TORCH else None,
    "cuda": (torch.version.cuda if HAS_TORCH else None),
    "gpu": (torch.cuda.get_device_name(0) if HAS_TORCH and torch.cuda.is_available() else "cpu"),
    "models": [{k: m[k] for k in ("name", "repo", "params", "base", "price")}
               for m in MODELS if m["name"] in RAW],
    "config": {k: v for k, v in globals().items()
               if k.isupper() and isinstance(v, (int, float, str, bool, list))},
    "file_sha256_16": files,
    "figures": sorted(os.listdir(os.path.join(RESULTS_DIR, "figures"))),
    "tables": sorted(os.listdir(os.path.join(RESULTS_DIR, "tables"))),
}
json.dump(manifest, open(os.path.join(RESULTS_DIR, "run_manifest.json"), "w"), indent=2, default=str)
open(os.path.join(RESULTS_DIR, "requirements_freeze.txt"), "w").write(freeze)
print(json.dumps(headline, indent=2))
print("\nwritten: headline_numbers.json, run_manifest.json, requirements_freeze.txt")

In [ ]:
# ── 9.2 · Auto-generated results paragraph (paste, then edit and verify) ──────
d_route = headline["routing_delta_pp"]
ci_route = paired_delta(policies["CV-route@1"][0], policies["best-single(CV)@1"][0], seed=31)[1]
d_wd, ci_wd = paired_delta(policies[RESAMPLE_NAME][0], policies[f"cascade@{DEPTH}"][0], seed=32)
easy_txt = ""
if easy_ks:
    be = max(np.mean([G[m][k] for k in easy_ks]) for m in MAIN)
    bh = max(np.mean([G[m][k] for k in keys if diff_of[k] in ("hard", "very_hard")] or [0])
             for m in MAIN)
    easy_txt = (f"The difficulty gradient is visible in the raw scores: the best model solves "
                f"{be*100:.1f}% of easy problems against {bh*100:.1f}% of hard ones, indicating "
                f"the low absolute numbers reflect benchmark difficulty rather than a harness "
                f"artifact. ")
para = f"""
RESULTS (auto-generated {datetime.datetime.now():%Y-%m-%d} -- verify every number before submitting)

We evaluate {len(RAW)} open-weight models ({len(MAIN)} instruction-tuned models in the routing
pool, {len(BASE)} base model(s) benchmarked separately) on {len(keys)} TACO problems spanning
{len(cats)} algorithmic categories, grading every provided test case rather than a fixed prefix.
Decoding is greedy for pass@1 and {N_SAMPLES} samples at T={SAMPLE_TEMP}/top-p={TOP_P} for the
unbiased pass@k estimator. {easy_txt}The strongest single model under {CV_FOLDS}-fold
cross-validation is {BS_NAME} at {headline['best_single_pass1']*100:.1f}% pass@1. Category-aware
routing, with the route map fit only on training folds, reaches
{headline['cv_route_pass1']*100:.1f}% (delta = {d_route:+.1f} pp, 95% CI
[{ci_route[0]*100:+.1f}, {ci_route[1]*100:+.1f}] pp). A per-problem oracle over the same pool
reaches {headline['oracle_pass1']*100:.1f}% ({headline['oracle_gap_pp']:+.1f} pp over the best
single model), so the models are complementary even where category-level routing fails to
exploit that complementarity. At a matched budget, resampling the best single model {RESAMPLE_K} times
solves {headline[f'resample{RESAMPLE_K}_best_single']*100:.1f}% while a cheapest-first cascade gated on
{PUBLIC_GATE_K} public test cases solves {headline[f'cascade@{DEPTH}']*100:.1f}%
(delta = {d_wd*100:+.1f} pp, 95% CI [{ci_wd[0]*100:+.1f}, {ci_wd[1]*100:+.1f}] pp).
"""
open(os.path.join(RESULTS_DIR, "results_paragraph.txt"), "w").write(para)
print(para)

## 9.3 · Artifact → paper element

| Artifact | Where it goes |
|---|---|
| `tables/tab1_models` | Setup: models, precision, prices |
| `tables/tab2_overall` | Results: headline pass@k with CIs and partial-credit case fractions |
| `tables/tab2b_difficulty` | Results: difficulty gradient — the evidence that low scores are the benchmark |
| `tables/tab2c_errors`, `fig7` | Results: failure composition — the evidence that the harness is sound |
| `tables/tab3_category_matrix`, `fig1`, `fig2` | Results: per-category performance |
| `tables/tab4_pairwise_stats` | Results: pairwise significance (BH + Holm) |
| `tables/tab5_policies`, `fig3`, `fig4`, `fig5` | Analysis: CV routing and width-vs-depth at matched budget |
| `tables/tab6_cost_sensitivity` | Analysis: cost-model robustness + wall-clock axis |
| `tables/tab7_repair` | Analysis: repair vs blind resample |
| `tables/tab8_taxonomy_sensitivity` | Setup/Analysis: is the category axis an artifact of the priority order? |
| `tables/tab9_route_stability` | Analysis: does a stable route map exist? |
| `fig6` | Analysis: complementarity / oracle gap |
| `headline_numbers.json`, `results_paragraph.txt` | First draft of the results section |
| `run_manifest.json`, `gen/*.jsonl`, `grade/*.jsonl` | Reproducibility appendix + released artifacts |

## 9.4 · What you may claim, and what you may not

**Claimable, if the numbers support it**
- "Cross-validated category routing does / does not outperform the best single model on TACO
  (Δ = X pp, 95% CI [..])."
- "At a matched budget, resampling one model dominates / does not dominate a gated cascade."
- "Error-feedback repair does / does not beat blind resampling at equal budget."
- "Models are complementary: a per-problem oracle exceeds the best single model by X pp."
- "Routing conclusions are / are not robust to the taxonomy priority order (Table 8)."
- "The same three conclusions replicate / fail to replicate on HumanEval+" — this is the
  strongest sentence in the paper, and it needs both notebooks run to completion.

**Not claimable**
- Any number from a TRIAL / QUICK / DRY run — the notebook banners these.
- Any comparison with your old `benchmark_results/` numbers. Those used 5-case grading and a
  sampled pass@1; they measure something else.
- Any routing number without the CV prefix.
- Cascade numbers described as if the gate were free information — say plainly that the gate
  uses `PUBLIC_GATE_K` cases and that the same cases are also part of the grading set.
- Anything about base models (StarCoder2) being "worse at category X": they are prompted in
  completion mode and excluded from routing for exactly that reason.

## 9.5 · Limitations to write into the paper

1. **Contamination.** TACO is drawn from Codeforces, LeetCode and similar sources that are
   almost certainly in these models' pretraining data. State it in the abstract.
2. **Low absolute scores.** Report Table 2b and 2c to establish that this is benchmark
   difficulty rather than a harness artifact, and acknowledge that binary pass@1 near the floor
   gives McNemar very little power — Table 4 will show few discordant pairs.
3. **Many-to-one taxonomy.** Report the multi-tag rate from §1.3 and the permutation
   sensitivity from §7.6. Single-label assignment stays a modelling choice.
4. **Test-case capping.** Grading uses up to `MAX_TEST_CASES` cases; the pre-cap count is
   recorded per problem. A solution passing 10 cases is not proven correct.
5. **Public/private gate.** Gating on the first cases is deploy-realistic but those cases are
   also in the grading set. Report the gate-pass rate and the solve rate conditional on
   passing the gate, both printed in §7.2.
6. **Cost model.** Relative per-token prices under local sequential serving, excluding model
   load time — which penalises multi-model policies in reality. Hence the wall-clock axis.
7. **Single sampling seed.** The bootstrap captures problem variance, not seed variance.

## 9.6 · Pre-submission checklist

- [ ] HF token rotated; no secrets in any notebook, output file or git history
- [ ] Full run: no PARTIAL banner, `TRIAL_MODE`/`QUICK_TEST`/`DRY_RUN` all False
- [ ] Table 2b shows a real difficulty gradient (if not, debug prompting before writing)
- [ ] Table 2c failures dominated by wrong_answer / tle, not syntax_error / missing_fn
- [ ] Route-map stability (Table 9) and taxonomy sensitivity (Table 8) both reported
- [ ] All 7 figures and 9 tables regenerated from the final checkpoints
- [ ] Companion HumanEval+ notebook run with the same protocol; both results sections written
- [ ] Related work: RouterBench, FrugalGPT, RouteLLM, Large Language Monkeys, TACO, Self-Refine
- [ ] `gen/*.jsonl` and `grade/*.jsonl` released so others can re-grade without a GPU